## Preprocess Data the Male Training Data

In [ ]:
import pandas as pd

# File paths
base_path = '<LOCAL_PATH>'

# Define the datasets
datasets = {
    '(Train)': 'train_combined.csv',
    '(Test)': 'test_combined.csv'
}

# Define function to print unique IDs and status
def print_unique_ids_and_status(df, label):
    ids = sorted(df['id'].unique())
    print(f"\n📄 {label} — {len(ids)} unique IDs:")
    print(ids)

    # CWD Positive
    positive_ids = sorted(df.loc[df['CWD_status'] == 'positive', 'id'].unique())
    print(f"  ➡ CWD Positive IDs ({len(positive_ids)}): {positive_ids}")

    # CWD Negative
    negative_ids = sorted(df.loc[df['CWD_status'] == 'negative', 'id'].unique())
    print(f"  ➡ CWD Negative IDs ({len(negative_ids)}): {negative_ids}")

def run_datetime_eda(df, label):
    print(f"\n🧪 EDA on `datetime` for {label}")

    # Check for missing
    null_count = df['datetime'].isnull().sum()
    print(f"  ➤ Null values in `datetime`: {null_count}")

    # Convert to datetime
    df['datetime'] = pd.to_datetime(df['datetime'], errors='coerce')
    coercion_count = df['datetime'].isnull().sum() - null_count
    if coercion_count > 0:
        print(f"  ⚠️ {coercion_count} additional nulls after conversion (possibly bad formats)")

    # Check range
    min_time = df['datetime'].min()
    max_time = df['datetime'].max()
    print(f"  ➤ Overall date range: {min_time} to {max_time}")

    # Duration per ID
    id_group = df.groupby('id')['datetime']
    durations = (id_group.max() - id_group.min()).dt.total_seconds() / (60*60*24)
    print(f"  ➤ Duration (days) across IDs: min={durations.min():.2f}, max={durations.max():.2f}, median={durations.median():.2f}")

    # Continuity: time between rows per ID
    discontinuous_ids = []
    for _id, group in df.sort_values('datetime').groupby('id'):
        times = group['datetime'].dropna()
        deltas = times.diff().dt.total_seconds().dropna()
        if (deltas == 0).all():
            discontinuous_ids.append(_id)

    if discontinuous_ids:
        print(f"  ⚠️ {len(discontinuous_ids)} IDs have no time variation (i.e., all timestamps are identical):")
        print(f"     {discontinuous_ids[:10]}{' ...' if len(discontinuous_ids) > 10 else ''}")
    else:
        print(f"  ✅ All IDs have time variation.")

    return df

# Load and process each dataset
for label, file_name in datasets.items():
    path = f"{base_path}/{file_name}"
    df = pd.read_csv(path)
    print_unique_ids_and_status(df, label)
    df = run_datetime_eda(df, label)


## Preprocess cAE no CV

In [ ]:
import pandas as pd
import numpy as np
import joblib
from pathlib import Path
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder

# ── CONFIG ─────────────────────────────────────────────────────────────────────
base = Path("<LOCAL_PATH>")
train_raw  = base/"train_combined.csv"
test_raw   = base/"test_combined.csv"

out_train  = base/"train_processed_infection_30_Conditional_exp.csv"
out_val    = base/"val_processed_infection_30_Conditional_exp.csv"
out_test   = base/"test_processed_infection_30_Conditional_exp.csv"

enc_path   = base/"encoder_infection_30_Conditional_exp.pkl"
sca_path   = base/"scaler_infection_30_Conditional_exp.pkl"

# ── DROP LOCATION & NON-MOVEMENT FIELDS ───────────────────────────────────────
to_drop = [
    'x_2','x_1','x_2_2','transition_to_migratory','id_1','phases',
    'migration_class_grouped','hour','movement_group','r2n_smoothed',
    'suspect_jump','extreme_jump','SATS','rel.angle','utm_easting','utm_northing',
    'dx','dy','dt','cum_dist','dist_per_hr','rolling_r2n_sd','rolling_r2n_mean',
    'rolling_sinuosity_sd','r2n','X','x','y','burst','special_behavior','sats','dop',
    'latitude', 'longitude', 'altitude', 'CWD_positive',
]

# ── PRIMARY MOVEMENT METRICS ──────────────────────────────────────────────────
num_feats = [
    'dist','velo','abs_rel_angle','sinuosity_28pt',
    'rolling_abs_rel_angle_mean','rolling_abs_rel_angle_sd',
    'rolling_abs_rel_angle_skew','rolling_abs_rel_angle_kurt',
    'rolling_velo_mean','rolling_velo_sd','rolling_velo_skew','rolling_velo_kurt',
    'low','est','high'
]

cat_feats  = ['season']
meta_feats = ['pkey','id','date_time','CWD_status']   # ← now includes pkey

# ── DEFINE STAGE LABELING ─────────────────────────────────────────────────────
early_cwd_ids = [6, 11, 19, 20, 25, 28, 32, 39, 40, 48, 62, 74, 105, 122, 125]
late_cwd_ids  = [1, 2, 8, 12, 15, 16, 17, 18, 21, 23, 24, 30, 31, 35, 36, 37, 57, 58, 61,
                 64, 65, 67, 68, 69, 70, 71, 72, 79, 83, 90, 91, 93, 94, 96, 102, 113, 114, 121, 130, 141]

def assign_stage_label(row):
    if str(row['CWD_status']).lower() == 'positive':   # only assign if positive
        if row['id'] in early_cwd_ids:
            return 1  # early
        elif row['id'] in late_cwd_ids:
            return 2  # late
    return 0  # negative or unlabeled

# ── PREPROCESS FUNCTION ───────────────────────────────────────────────────────
def preprocess(df, encoder=None, scaler=None, fit=False, add_label_column=True):
    # Drop unused columns and missing data
    df = df.drop(columns=to_drop, errors='ignore')
    df = df.dropna(subset=num_feats + cat_feats).copy()

    # Time ordering
    df['date_time'] = pd.to_datetime(df['date_time'], errors='coerce')
    df = df.sort_values(['id', 'date_time']).reset_index(drop=True)

    # Keep pkey + id + date_time + status
    meta = df[meta_feats].copy()

    # ─ Categorical encoding
    if fit:
        encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
        cat_mat = encoder.fit_transform(df[cat_feats])
    else:
        cat_mat = encoder.transform(df[cat_feats])
    cat_names = encoder.get_feature_names_out(cat_feats).tolist()

    # ─ Scaling
    if fit:
        scaler = MinMaxScaler()
        num_mat = scaler.fit_transform(df[num_feats])
    else:
        num_mat = scaler.transform(df[num_feats])

    # Assemble features
    X = np.hstack([num_mat, cat_mat])
    feats = num_feats + cat_names
    feat_df = pd.DataFrame(X, columns=feats)

    # ─ Combine meta + features
    full = pd.concat([meta.reset_index(drop=True), feat_df.reset_index(drop=True)], axis=1)

    # Labels
    if add_label_column and 'CWD_status' in full.columns:
        full['CWD_label']  = (full['CWD_status'].astype(str).str.lower() == 'positive').astype(np.float32)
        full['stage_label'] = full.apply(assign_stage_label, axis=1)

    return full, encoder, scaler

# ── SPLIT FEATURES + LABELS ───────────────────────────────────────────────────
def split_features_labels(df, is_test=False):
    feature_cols = [c for c in df.columns if c not in ('pkey','id','date_time','CWD_status','CWD_label','stage_label')]
    X = df[feature_cols].values.astype(np.float32)
    if is_test:
        return X
    else:
        y = df[['CWD_label','stage_label']].values
        return X, y

# ── 1) LOAD + PROCESS TRAINING DATA ───────────────────────────────────────────
df_train_raw = pd.read_csv(train_raw)
df_full_train, enc, sca = preprocess(df_train_raw, fit=True, add_label_column=True)

# Save artifacts
joblib.dump(enc, enc_path)
joblib.dump(sca, sca_path)

# ── ID-disjoint split ─────────────────────────────────────────────────────────
unique_ids = df_full_train['id'].dropna().unique()
rng = np.random.default_rng(42)
rng.shuffle(unique_ids)

val_id_count = int(0.2 * len(unique_ids))
val_ids = unique_ids[:val_id_count]
train_ids = unique_ids[val_id_count:]

df_train = df_full_train[df_full_train['id'].isin(train_ids)].reset_index(drop=True)
df_val   = df_full_train[df_full_train['id'].isin(val_ids)].reset_index(drop=True)

df_train.to_csv(out_train, index=False)
df_val.to_csv(out_val, index=False)

# ── 2) TEST TRANSFORM ─────────────────────────────────────────────────────────
df_test_raw = pd.read_csv(test_raw)
df_test_proc, _, _ = preprocess(df_test_raw, encoder=enc, scaler=sca, fit=False, add_label_column=True)
df_test_proc.to_csv(out_test, index=False)

# ── 3) CONFIRMATION ───────────────────────────────────────────────────────────
print("✅ Preprocessing complete (pkey, status, binary + stage labels preserved).")
print(" → train IDs:", sorted(df_train['id'].dropna().unique().tolist())[:10], "… (#", df_train['id'].nunique(), ")")
print(" →   val IDs:", sorted(df_val['id'].dropna().unique().tolist())[:10],   "… (#", df_val['id'].nunique(),   ")")
print(" →  test IDs:", sorted(df_test_proc['id'].dropna().unique().tolist())[:10], "… (#", df_test_proc['id'].nunique(), ")")

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Conditional Autoencoder (cAE) — reviewer-ready training script
#  - Conditions on final status (CWD_label)
#  - Reproducible seeds + deterministic ops
#  - tf.data with cache/prefetch
#  - Checkpointing + history + metadata artifacts
#  - Clear, scaled-MSE loss (α = 1 + λ·d)
# ──────────────────────────────────────────────────────────────────────────────
import os, random
from pathlib import Path

import numpy as np
import pandas as pd
import joblib
import tensorflow as tf
from tensorflow.keras import layers, Model, regularizers
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
import matplotlib.pyplot as plt

# ===== Reproducibility =====
os.environ["PYTHONHASHSEED"] = "0"
random.seed(13); np.random.seed(13)
tf.keras.utils.set_random_seed(13)
try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass

# ===== Paths =====
base = Path("<LOCAL_PATH>")
train_path = base / "train_processed_infection_30_Conditional_exp.csv"
val_path   = base / "val_processed_infection_30_Conditional_exp.csv"
test_path  = base / "test_processed_infection_30_Conditional_exp.csv"

# outputs / artifacts
model_path       = base / "conditional_autoencoder_model_Conditional_EXP.h5"
ckpt_path        = base / "conditional_autoencoder_ckpt.weights.h5"
history_path     = base / "training_history.npy"
feat_cols_path   = base / "cae_feature_columns.pkl"
train_meta_path  = base / "cae_training_metadata.pkl"

# ===== Hyperparameters =====
batch_size  = 128
patience    = 5
learning_rate = 1e-5
l2_reg      = 1e-4         # set to 1e-4 if you want L2 regularization
LAMBDA_DEV  = 0.75          # scales MSE via α = 1 + λ·d
EPOCHS      = 100
SHUFFLE_BUF = 10_000

# ===== Load data =====
def load_df(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    if "date_time" in df.columns:
        df["date_time"] = pd.to_datetime(df["date_time"], errors="coerce")
    return df

df_train = load_df(train_path)
df_val   = load_df(val_path)
df_test  = load_df(test_path)

# ===== Split features / labels =====
def split_features_labels(df: pd.DataFrame):
    """
    Returns X (float32), y (float32, shape [N,1]).
    Excludes metadata/label columns from features to avoid leakage.
    """
    # Metadata / labels to exclude from features (robust against presence/absence)
    exclude = {'pkey','id','date_time','CWD_status','CWD_label','stage_label'}
    feature_cols = [c for c in df.columns if c not in exclude]

    # Build labels (final status)
    if 'CWD_label' in df.columns:
        y = df['CWD_label'].astype(np.float32).values.reshape(-1, 1)
    elif 'CWD_status' in df.columns:
        label_map = {'negative': 0.0, 'positive': 1.0}
        y = df['CWD_status'].map(label_map).astype(np.float32).values.reshape(-1, 1)
    else:
        raise ValueError("No label column found (expected 'CWD_label' or 'CWD_status').")

    X = df[feature_cols].values.astype(np.float32)
    return X, y, feature_cols

X_train, y_train, feat_cols = split_features_labels(df_train)
X_val,   y_val,   _         = split_features_labels(df_val)
X_test,  y_test,  _         = split_features_labels(df_test)

# Freeze feature list for reproducibility
joblib.dump(feat_cols, feat_cols_path)
print(f"✅ Frozen feature order saved → {feat_cols_path}")

# Label sanity
for name, y in [("train", y_train), ("val", y_val), ("test", y_test)]:
    uniq = np.unique(y)
    print(f"[{name}] shape={y.shape} | CWD_label counts → 0: {(y==0).sum():,} | 1: {(y==1).sum():,} | uniq={uniq}")

print(f"Features shape (train): {X_train.shape} | (val): {X_val.shape}")

# ===== tf.data pipelines =====
AUTOTUNE = tf.data.AUTOTUNE

def make_ds(X, y, training=False):
    ds = tf.data.Dataset.from_tensor_slices(((X, y), X))
    if training:
        ds = ds.shuffle(min(SHUFFLE_BUF, len(X)), seed=13, reshuffle_each_iteration=True)
    ds = ds.batch(batch_size, drop_remainder=False).cache().prefetch(AUTOTUNE)
    return ds

train_ds = make_ds(X_train, y_train, training=True)
val_ds   = make_ds(X_val,   y_val,   training=False)

# ===== Model =====
input_dim = X_train.shape[1]
reg = regularizers.l2(l2_reg) if l2_reg and l2_reg > 0 else None

# Inputs
encoder_input_feat  = layers.Input(shape=(input_dim,), name="features")
encoder_input_label = layers.Input(shape=(1,),         name="cwd_label")

# Encoder
x = layers.Concatenate()([encoder_input_feat, encoder_input_label])
x = layers.Dense(64, activation='relu', kernel_regularizer=reg)(x)
x = layers.Dense(32, activation='relu', kernel_regularizer=reg)(x)
encoded = layers.Dense(8, activation='relu', kernel_regularizer=reg, name="latent")(x)
encoder_model = Model([encoder_input_feat, encoder_input_label], encoded, name='encoder')

# Decoder
decoder_input_latent = layers.Input(shape=(8,), name="latent_in")
decoder_input_label  = layers.Input(shape=(1,), name="cwd_label_in")
y = layers.Concatenate()([decoder_input_latent, decoder_input_label])
y = layers.Dense(32, activation='relu', kernel_regularizer=reg)(y)
y = layers.Dense(64, activation='relu', kernel_regularizer=reg)(y)
decoded = layers.Dense(input_dim, activation='linear')(y)
decoder_model = Model([decoder_input_latent, decoder_input_label], decoded, name='decoder')

# Full cAE
z = encoder_model([encoder_input_feat, encoder_input_label])
recon = decoder_model([z, encoder_input_label])
autoencoder = Model([encoder_input_feat, encoder_input_label], recon, name='conditional_autoencoder')

# ===== Loss (scaled-MSE; clearer reviewer mapping to MSE+λ·SSE) =====
LOSS_SCALE = 1.0 + LAMBDA_DEV * input_dim  # α = 1 + λ·d

def get_custom_loss(loss_scale=LOSS_SCALE):
    def custom_loss(y_true, y_pred):
        mse = tf.reduce_mean(tf.square(y_true - y_pred), axis=-1)  # mean over features
        return loss_scale * mse
    return custom_loss

loss_fn = get_custom_loss(LOSS_SCALE)
autoencoder.compile(optimizer=tf.keras.optimizers.Adam(learning_rate), loss=loss_fn)

# ===== Train =====
early_stop = EarlyStopping(monitor="val_loss", patience=patience, restore_best_weights=True)
ckpt = ModelCheckpoint(filepath=str(ckpt_path), save_weights_only=True, monitor="val_loss", save_best_only=True)

history = autoencoder.fit(
    train_ds,
    epochs=EPOCHS,
    validation_data=val_ds,
    callbacks=[early_stop, ckpt],
    verbose=1
)

# Save model + history
autoencoder.save(model_path)
np.save(history_path, history.history, allow_pickle=True)
print(f"✅ Model saved → {model_path}")
print(f"✅ History saved → {history_path}")
print(f"✅ Best weights checkpoint → {ckpt_path}")

# ===== Reconstruction errors (train/val) =====
recon_train = autoencoder.predict((X_train, y_train), batch_size=batch_size, verbose=0)
train_errors = loss_fn(X_train, recon_train).numpy().astype(np.float64)

recon_val = autoencoder.predict((X_val, y_val), batch_size=batch_size, verbose=0)
val_errors = loss_fn(X_val, recon_val).numpy().astype(np.float64)

def summarize_errors(errors, labels, name="split"):
    neg = errors[labels.flatten() == 0]
    pos = errors[labels.flatten() == 1]
    print(f"[{name}] errors: mean±sd (neg) = {neg.mean():.6g} ± {neg.std():.6g} | "
          f"(pos) = {pos.mean():.6g} ± {pos.std():.6g} | N={len(errors):,}")

summarize_errors(train_errors, y_train, "train")
summarize_errors(val_errors,   y_val,   "val")

# Optional: quick visualization (train by label)
try:
    plt.figure(figsize=(6, 3))
    plt.hist(train_errors[y_train.flatten() == 0], bins=50, alpha=0.6, label="Negative")
    plt.hist(train_errors[y_train.flatten() == 1], bins=50, alpha=0.6, label="Positive")
    plt.title("Reconstruction Error (train) by CWD Label")
    plt.xlabel("Scaled MSE")
    plt.ylabel("Frequency")
    plt.legend()
    plt.tight_layout()
    plt.show()
except Exception as e:
    print(f"(Histogram skipped: {e})")

# ===== Persist minimal training metadata for downstream reproducibility =====
train_meta = {
    "feature_columns": feat_cols,
    "input_dim": int(input_dim),
    "lambda_dev": float(LAMBDA_DEV),
    "loss_scale": float(LOSS_SCALE),   # α
    "batch_size": int(batch_size),
    "learning_rate": float(learning_rate),
    "epochs_ran": int(len(history.history.get("loss", []))),
    "early_stopped": bool("val_loss" in history.history),
    "best_ckpt_path": str(ckpt_path),
    "model_path": str(model_path),
    "seed": 13,
}
joblib.dump(train_meta, train_meta_path)
print(f"✅ Training metadata saved → {train_meta_path}")


## Validation Freezing

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# cAE VALIDATION — freeze slightly stricter ID τ on mean(margin) (FINAL status)
#  - flip-aware mean(margin) score (higher = more CWD+)
#  - stricter FPR target on negatives
#  - small positive nudge above the quantile
#  - calibrate on stable negatives only (min rows)
#  - SAVE a test-time min-rows eligibility gate (no test re-tuning)
# ──────────────────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import joblib
import tensorflow as tf
from tensorflow.keras.models import load_model
from pathlib import Path

# ===== Config / Paths =====
base = Path("<LOCAL_PATH>")
val_csv         = base / "val_processed_infection_30_Conditional_exp.csv"
train_meta_path = base / "cae_training_metadata.pkl"
feat_cols_path  = base / "cae_feature_columns.pkl"
out_path        = base / "cae_id_rule_params.pkl"

# ── Strictness knobs (your best result) ───────────────────────────────────────
ID_FPR_TARGET      = 0.02     # right-tail on NEG IDs
NUDGE_STD_FRACTION = 0.10     # + 0.10 * std(neg_scores)
MIN_ROWS_FOR_CAL   = 1      # use only neg IDs with ≥ this many rows to calibrate τ

# 🔒 New: freeze a test-time eligibility gate (to suppress small-N FPs)
MIN_ROWS_FOR_TEST  = 1

# ===== Load training metadata & artifacts =====
meta       = joblib.load(train_meta_path)
model_path = Path(meta["model_path"])
lambda_dev = float(meta["lambda_dev"])
loss_scale = float(meta["loss_scale"])
feat_cols  = joblib.load(feat_cols_path)

# ===== Load model & loss =====
model = load_model(model_path, compile=False)
def loss_fn(y_true, y_pred, scale=loss_scale):
    mse = tf.reduce_mean(tf.square(y_true - y_pred), axis=-1)
    return scale * mse

# ===== Load validation (frozen features) =====
df = pd.read_csv(val_csv)
if "date_time" in df.columns:
    df["date_time"] = pd.to_datetime(df["date_time"], errors="coerce")
if "CWD_label" not in df.columns:
    raise ValueError("CWD_label not found in validation CSV.")

X_val = df[feat_cols].values.astype(np.float32)
ids   = df["id"].values
y_row = df["CWD_label"].astype(int).values

# ===== Dual reconstructions & margin =====
N = X_val.shape[0]
zeros = np.zeros((N,1), dtype=np.float32)
ones  = np.ones((N,1), dtype=np.float32)
recon_neg = model.predict((X_val, zeros), batch_size=256, verbose=0)
recon_pos = model.predict((X_val, ones ), batch_size=256, verbose=0)

err_neg = loss_fn(X_val, recon_neg).numpy().astype(np.float64).flatten()
err_pos = loss_fn(X_val, recon_pos).numpy().astype(np.float64).flatten()
margin  = err_neg - err_pos

rows = pd.DataFrame({"id": ids, "margin": margin, "y": y_row})

# ===== ID-level stats & flip-aware score =====
g = rows.groupby("id", sort=False)
id_mean_margin = g["margin"].mean()
id_n_rows      = g.size()
id_label       = g["y"].max().astype(int)

neg_mask = (id_label == 0)
pos_mask = (id_label == 1)

neg_mean = float(id_mean_margin[neg_mask].mean()) if neg_mask.any() else np.nan
pos_mean = float(id_mean_margin[pos_mask].mean()) if pos_mask.any() else np.nan

# Flip so that "higher score = more CWD+"
if np.isfinite(neg_mean) and np.isfinite(pos_mean) and (pos_mean < neg_mean):
    score_name = "minus_mean_margin"
    id_score = -id_mean_margin
else:
    score_name = "mean_margin"
    id_score = id_mean_margin

# ===== Calibrate τ on NEGATIVE IDs (optionally stable only) =====
neg_scores_all = id_score[neg_mask]
neg_rows_all   = id_n_rows[neg_mask]
stable_neg     = (neg_rows_all >= MIN_ROWS_FOR_CAL)
cal_base       = neg_scores_all[stable_neg] if stable_neg.any() else neg_scores_all
if len(cal_base) == 0:
    raise ValueError("No negative IDs available to calibrate τ.")

tau_q   = float(np.quantile(cal_base.values, 1.0 - ID_FPR_TARGET))
tau_std = float(cal_base.values.std(ddof=1)) if len(cal_base) > 1 else 0.0
tau     = tau_q + NUDGE_STD_FRACTION * tau_std

# ===== Validation metrics at frozen τ =====
id_pred = (id_score > tau).astype(int)

tp = int(((id_pred == 1) & (id_label == 1)).sum())
fp = int(((id_pred == 1) & (id_label == 0)).sum())
fn = int(((id_pred == 0) & (id_label == 1)).sum())
tn = int(((id_pred == 0) & (id_label == 0)).sum())
prec = tp / (tp + fp) if (tp + fp) else 0.0
rec  = tp / (tp + fn) if (tp + fn) else 0.0
fpr  = fp / (fp + tn) if (fp + tn) else 0.0

print(f"\n[mean-margin policy] score='{score_name}'")
print(f"  neg_mean={neg_mean:.6g}, pos_mean={pos_mean:.6g}")
print(f"  τ_quantile={tau_q:.6g} (target FPR={ID_FPR_TARGET:.3f}, "
      f"base={'stable negs' if stable_neg.any() else 'all negs'})")
print(f"  τ_final   ={tau:.6g} (+{NUDGE_STD_FRACTION:.2f}·std={tau_std:.6g})")
print(f"  VAL ID metrics → Precision={prec:.3f} | Recall={rec:.3f} | FPR={fpr:.3f} | TP={tp} FP={fp} FN={fn} TN={tn}")

# Row-count snapshot (for reviewers + to justify MIN_ROWS_FOR_TEST)
print("\nRow-count per ID (VAL):")
print(pd.Series(id_n_rows).describe().to_string())

# Persist frozen policy
frozen = {
    "decision_level": "id_mean_margin",
    "score_name": score_name,                 # "mean_margin" or "minus_mean_margin"
    "tau_id_score": float(tau),
    "id_fpr_target": float(ID_FPR_TARGET),
    "lambda_dev": float(lambda_dev),
    "input_dim": int(meta["input_dim"]),
    "loss_scale": float(loss_scale),
    "model_path": str(model_path),
    "feature_columns": feat_cols,
    "calibrated_on": "stable_negatives" if stable_neg.any() else "all_negatives",
    "min_rows_for_cal": int(MIN_ROWS_FOR_CAL),
    # 🔒 Save the test-time eligibility gate (no test re-tuning)
    "min_rows_for_test": int(MIN_ROWS_FOR_TEST),
}
joblib.dump(frozen, out_path)
print(f"\n✅ Saved stricter mean-margin policy → {out_path}")


In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# cAE TESTING — apply frozen mean-margin τ with min-rows gate (FINAL status)
# + Reviewer-ready metrics on the eligible set
# ──────────────────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import joblib
import tensorflow as tf
from tensorflow.keras.models import load_model
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    confusion_matrix, precision_recall_fscore_support
)
from pathlib import Path

# ===== Paths =====
base = Path("<LOCAL_PATH>")
test_csv        = base / "test_processed_infection_30_Conditional_exp.csv"
train_meta_path = base / "cae_training_metadata.pkl"
feat_cols_path  = base / "cae_feature_columns.pkl"
policy_path     = base / "cae_id_rule_params.pkl"

rows_out_csv = base / "test_rows_cae_meanmargin.csv"
ids_out_csv  = base / "test_ids_cae_meanmargin.csv"

# ===== Load artifacts =====
meta      = joblib.load(train_meta_path)
feat_cols = joblib.load(feat_cols_path)
policy    = joblib.load(policy_path)

model_path    = Path(policy.get("model_path", meta["model_path"]))
lambda_dev    = float(policy.get("lambda_dev", meta["lambda_dev"]))
loss_scale    = float(policy.get("loss_scale", meta["loss_scale"]))
score_name    = policy.get("score_name", "mean_margin")  # "mean_margin" or "minus_mean_margin"
tau           = float(policy["tau_id_score"])
min_rows_test = int(policy.get("min_rows_for_test", 0))

print(f"Policy: id_mean_margin | score={score_name} | τ={tau:.6g} | min_rows_test={min_rows_test}")
print(f"Model: {model_path.name} | λ={lambda_dev} | α={loss_scale:.6g}")

# ===== Load model & loss =====
model = load_model(model_path, compile=False)
def loss_fn(y_true, y_pred, scale=loss_scale):
    mse = tf.reduce_mean(tf.square(y_true - y_pred), axis=-1)
    return scale * mse

# ===== Load test data (frozen features) =====
df = pd.read_csv(test_csv)
if "date_time" in df.columns:
    df["date_time"] = pd.to_datetime(df["date_time"], errors="coerce")

missing = [c for c in feat_cols if c not in df.columns]
if missing:
    raise KeyError(f"Test CSV missing required features: {missing[:10]}{' ...' if len(missing)>10 else ''}")

X   = df[feat_cols].values.astype(np.float32)
ids = df["id"].values
tst = df["date_time"] if "date_time" in df.columns else None
y_gt = df["CWD_label"].astype(int).values if "CWD_label" in df.columns else None

# ===== Dual reconstructions & row margin =====
N = X.shape[0]
zeros = np.zeros((N,1), dtype=np.float32)
ones  = np.ones((N,1), dtype=np.float32)
recon_neg = model.predict((X, zeros), batch_size=256, verbose=0)
recon_pos = model.predict((X, ones ), batch_size=256, verbose=0)

err_neg = loss_fn(X, recon_neg).numpy().astype(np.float64).flatten()
err_pos = loss_fn(X, recon_pos).numpy().astype(np.float64).flatten()
margin  = err_neg - err_pos  # >0 ⇒ fits CWD+ manifold better

rows = pd.DataFrame({
    "id": ids, "err_as_neg": err_neg, "err_as_pos": err_pos, "margin": margin
})
if tst is not None: rows["date_time"] = tst
if y_gt is not None: rows["true_label"] = y_gt
rows.to_csv(rows_out_csv, index=False)
print(f"✅ Row outputs → {rows_out_csv}")

# ===== ID-level aggregation & decision =====
g = rows.groupby("id", sort=False)
id_mean_margin  = g["margin"].mean()
id_mean_err_pos = g["err_as_pos"].mean()
id_frac_pos     = g["margin"].apply(lambda s: (s > 0).mean())
id_n_rows       = g.size()

ids_df = pd.DataFrame({"id": id_mean_margin.index})
ids_df["mean_margin"]   = id_mean_margin.values
ids_df["mean_err_pos"]  = id_mean_err_pos.values
ids_df["frac_pos_rows"] = id_frac_pos.values
ids_df["n_rows"]        = id_n_rows.values

# flip-aware primary score (same convention as validation)
if score_name == "minus_mean_margin":
    ids_df["score"] = -ids_df["mean_margin"]
else:
    ids_df["score"] = ids_df["mean_margin"]

# eligibility gate (frozen from validation)
eligible = (ids_df["n_rows"].values >= min_rows_test) if min_rows_test > 0 else np.ones(len(ids_df), dtype=bool)
ids_df["eligible"] = eligible.astype(int)

# frozen decision
ids_df["id_pred"] = ((ids_df["score"] > tau) & eligible).astype(int)

# attach GT at ID level (max row label per ID)
if y_gt is not None:
    id_true = rows.groupby("id", sort=False)["true_label"].max().reindex(ids_df["id"]).astype(int).reset_index(drop=True)
    ids_df["true_label_id"] = id_true.values

ids_df.to_csv(ids_out_csv, index=False)
print(f"✅ ID outputs → {ids_out_csv}")

flagged = ids_df.loc[ids_df["id_pred"] == 1, "id"].tolist()
print("\n🧠 IDs deemed CWD+ under mean-margin policy + min-rows gate:")
print(flagged if len(flagged) <= 50 else flagged[:50] + ['... (truncated)'])
print(f"Total flagged IDs: {len(flagged)} / {len(ids_df)}")

# ===== Table metrics (eligible set only) =====
def wilson_ci(k, n, z=1.96):
    if n == 0: return (np.nan, np.nan)
    p = k / n
    denom = 1 + z*z/n
    center = (p + z*z/(2*n)) / denom
    half = z * np.sqrt((p*(1-p)/n) + (z*z/(4*n*n))) / denom
    return (center - half, center + half)

if "true_label_id" in ids_df.columns:
    eval_df = ids_df[ids_df["eligible"] == 1].copy()
    y_true  = eval_df["true_label_id"].astype(int).to_numpy()
    y_pred  = eval_df["id_pred"].astype(int).to_numpy()

    from sklearn.metrics import confusion_matrix, precision_recall_fscore_support
    TN, FP, FN, TP = confusion_matrix(y_true, y_pred, labels=[0,1]).ravel()

    prec, rec, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="binary", zero_division=0
    )
    spec = TN/(TN+FP) if (TN+FP) else np.nan
    sens = TP/(TP+FN) if (TP+FN) else np.nan  # = recall
    bal_acc = 0.5*(sens + spec) if np.isfinite(spec) and np.isfinite(sens) else np.nan
    fpr = FP/(FP+TN) if (FP+TN) else np.nan
    fnr = FN/(FN+TP) if (FN+TP) else np.nan
    acc = (TP+TN)/(TP+TN+FP+FN) if (TP+TN+FP+FN) else np.nan
    prevalence = y_true.mean() if len(y_true) else np.nan

    # 95% CIs (Wilson) for precision and recall
    prec_lo, prec_hi = wilson_ci(TP, TP+FP)
    rec_lo,  rec_hi  = wilson_ci(TP, TP+FN)

    # One-row table
    table = pd.DataFrame({
        "N_eval":[int(len(eval_df))],
        "TP":[int(TP)], "FP":[int(FP)], "FN":[int(FN)], "TN":[int(TN)],
        "prevalence":[prevalence],
        "precision":[prec], "precision_CI_lo":[prec_lo], "precision_CI_hi":[prec_hi],
        "recall":[rec],     "recall_CI_lo":[rec_lo],   "recall_CI_hi":[rec_hi],
        "F1":[f1],
        "specificity":[spec], "sensitivity":[sens],
        "balanced_accuracy":[bal_acc],
        "FPR":[fpr], "FNR":[fnr],
        "accuracy":[acc],
        # optional provenance
        "tau_pooled_raw":[pooled_tau_raw],
        "min_rows_test":[min_rows_test],
    })

    # Save + pretty print
    metrics_csv = ids_out_csv.with_name("test_metrics_table_RAW.csv")
    table.round(3).to_csv(metrics_csv, index=False)
    print(f"\n✅ Wrote metrics table → {metrics_csv}")
    print(table.to_string(index=False, float_format=lambda x: f'{x:.3f}'))

    # ===== Quick figure (confusion + P/R with 95% CIs) =====
    try:
        import matplotlib.pyplot as plt
        fig, axes = plt.subplots(1, 2, figsize=(9, 3.6))

        # Confusion matrix (rows=GT, cols=Pred)
        cm = np.array([[TN, FP],
                       [FN, TP]])
        im = axes[0].imshow(cm, cmap="Blues")
        for (i, j), v in np.ndenumerate(cm):
            axes[0].text(j, i, int(v), ha="center", va="center", color="black", fontsize=11)
        axes[0].set_xticks([0,1]); axes[0].set_yticks([0,1])
        axes[0].set_xticklabels(["Pred 0","Pred 1"])
        axes[0].set_yticklabels(["GT 0","GT 1"])
        axes[0].set_title("Confusion (eligible set)")
        fig.colorbar(im, ax=axes[0], fraction=0.046, pad=0.04)

        # Bars: Precision, Recall (with 95% CI), plus Spec/Acc/BalAcc
        labels = ["Precision", "Recall", "Specificity", "Accuracy", "Balanced Acc."]
        vals   = [prec,        rec,      spec,          acc,        bal_acc]
        axes[1].bar(range(len(vals)), vals, color=["#4C78A8","#F58518","#54A24B","#9D755D","#B279A2"])
        # error bars for P/R only
        yerr_low  = [prec - prec_lo,  rec - rec_lo,  0, 0, 0]
        yerr_high = [prec_hi - prec,  rec_hi - rec,  0, 0, 0]
        axes[1].errorbar(range(len(vals)), vals, yerr=[yerr_low, yerr_high],
                         fmt="none", ecolor="k", capsize=3, lw=1)
        axes[1].set_xticks(range(len(vals))); axes[1].set_xticklabels(labels, rotation=20)
        axes[1].set_ylim(0, 1)
        axes[1].axhline(prevalence, color="gray", ls="--", lw=1, label="GT prevalence")
        axes[1].legend()
        axes[1].set_title("Point metrics (95% CI on P/R)")
        fig.tight_layout()
        plt.show()
    except Exception as e:
        print(f"(Skipped plots: {e})")


# Diagnostics: flagged IDs
flagged = ids_df.loc[ids_df["id_pred"] == 1, "id"].tolist()
print("\n🧠 IDs flagged — RAW pooled:", flagged)
print(f"Total flagged: {len(flagged)} / {len(ids_df)}")


In [ ]:
# Define the actual CWD instances and unaffected CWD instances
actual_cwd_instances = [16, 18, 19, 21, 31, 53, 58, 72, 92, 102]
unaffected_cwd_instances = [3, 4, 7, 34, 38, 42, 43, 55, 56, 60, 77, 78, 82, 89, 95, 100, 111, 115, 119, 120, 129]


# You already have high_anomaly_ids from Step 1
print("High anomaly IDs:", flagged)

# Convert high_anomaly_ids to a set for easier comparison
high_deviation_ids = set(flagged)

# Calculate True Positives (TP), False Negatives (FN), False Positives (FP), and True Negatives (TN)
TP = len([ID for ID in actual_cwd_instances if ID in flagged])
FN = len([ID for ID in actual_cwd_instances if ID not in flagged])
FP = len([ID for ID in unaffected_cwd_instances if ID in flagged])
TN = len([ID for ID in unaffected_cwd_instances if ID not in flagged])

# Calculate False Positive Rate (FPR), False Negative Rate (FNR), and Accuracy
FPR = FP / (FP + TN) if (FP + TN) > 0 else 0
FNR = FN / (FN + TP) if (FN + TP) > 0 else 0
accuracy = (TP + TN) / (TP + TN + FP + FN) if (TP + TN + FP + FN) > 0 else 0

# Print the results
print(f"True Positives (TP): {TP}")
print(f"False Negatives (FN): {FN}")
print(f"False Positives (FP): {FP}")
print(f"True Negatives (TN): {TN}")
print(f"False Positive Rate (FPR): {FPR:.2f}")
print(f"False Negative Rate (FNR): {FNR:.2f}")
print(f"Accuracy: {accuracy:.2f}")


## Preprocessing data with CV

In [ ]:
import pandas as pd
import numpy as np
import joblib
from pathlib import Path
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder

# ── CONFIG ─────────────────────────────────────────────────────────────────────
base = Path("<LOCAL_PATH>")
train_raw  = base/"train_combined.csv"
test_raw   = base/"test_combined.csv"   # optional; only used if you also want a single final transform

# Cross-validation
CV_K = 5
RANDOM_SEED = 42
FOLDS_DIR = base / f"cv_folds_k{CV_K}"
FOLDS_DIR.mkdir(exist_ok=True, parents=True)

# Global artifacts (optional, for single final run or test transform)
global_enc_path = base/"encoder_infection_30_Conditional_exp.pkl"
global_sca_path = base/"scaler_infection_30_Conditional_exp.pkl"

# ── DROP LOCATION & NON-MOVEMENT FIELDS ───────────────────────────────────────
to_drop = [
    'x_2','x_1','x_2_2','transition_to_migratory','id_1','phases',
    'migration_class_grouped','hour','movement_group','r2n_smoothed',
    'suspect_jump','extreme_jump','SATS','rel.angle','utm_easting','utm_northing',
    'dx','dy','dt','cum_dist','dist_per_hr','rolling_r2n_sd','rolling_r2n_mean',
    'rolling_sinuosity_sd','r2n','X','x','y','burst','special_behavior','sats','dop',
    'latitude', 'longitude', 'altitude', 'CWD_positive',
]

# ── PRIMARY MOVEMENT METRICS ──────────────────────────────────────────────────
num_feats = [
    'velo','abs_rel_angle','sinuosity_28pt',
    'rolling_abs_rel_angle_mean','rolling_abs_rel_angle_sd',
    'rolling_abs_rel_angle_skew','rolling_abs_rel_angle_kurt',
    'rolling_velo_mean','rolling_velo_sd','rolling_velo_skew','rolling_velo_kurt',
    'low','est','high'
]
cat_feats  = ['season']
meta_feats = ['pkey','id','datetime','CWD_status']  # keep these around

# ── STAGE LABELS (unchanged) ──────────────────────────────────────────────────
early_cwd_ids = [6, 11, 19, 20, 25, 28, 32, 39, 40, 48, 62, 74, 105, 122, 125]
late_cwd_ids  = [1, 2, 8, 12, 15, 16, 17, 18, 21, 23, 24, 30, 31, 35, 36, 37, 57, 58, 61,
                 64, 65, 67, 68, 69, 70, 71, 72, 79, 83, 90, 91, 93, 94, 96, 102, 113, 114, 121, 130, 141]

def assign_stage_label(row):
    if str(row['CWD_status']).lower() == 'positive':   # only assign if positive
        if row['id'] in early_cwd_ids:
            return 1  # early
        elif row['id'] in late_cwd_ids:
            return 2  # late
    return 0  # negative or unlabeled

# ── HELPERS ───────────────────────────────────────────────────────────────────
def make_ohe(categories):
    # Keeps OHE columns identical across folds by fixing categories
    try:
        return OneHotEncoder(sparse_output=False, handle_unknown='ignore', categories=categories)
    except TypeError:
        # for older scikit-learn
        return OneHotEncoder(sparse=False, handle_unknown='ignore', categories=categories)

def preprocess(df, encoder=None, scaler=None, fit=False, fixed_categories=None, add_label_column=True):
    # Drop & clean
    df = df.drop(columns=to_drop, errors='ignore')
    df['datetime'] = pd.to_datetime(df['datetime'], errors='coerce')
    df = df.dropna(subset=num_feats + cat_feats).copy()
    df = df.sort_values(['id','datetime']).reset_index(drop=True)

    meta = df[meta_feats].copy()

    # Encoder (fixed categories → identical columns per fold)
    if fit:
        enc = make_ohe(fixed_categories) if fixed_categories is not None else make_ohe('auto')
        cat_mat = enc.fit_transform(df[cat_feats])
        encoder = enc
    else:
        cat_mat = encoder.transform(df[cat_feats])
    cat_names = encoder.get_feature_names_out(cat_feats).tolist()

    # Scaler (fit on training split only to avoid leakage)
    if fit:
        sca = MinMaxScaler()
        num_mat = sca.fit_transform(df[num_feats])
        scaler = sca
    else:
        num_mat = scaler.transform(df[num_feats])

    X = np.hstack([num_mat, cat_mat])
    feats = num_feats + cat_names
    feat_df = pd.DataFrame(X, columns=feats)

    full = pd.concat([meta.reset_index(drop=True), feat_df.reset_index(drop=True)], axis=1)

    if add_label_column and 'CWD_status' in full.columns:
        full['CWD_label']  = (full['CWD_status'].astype(str).str.lower() == 'positive').astype(np.float32)
        full['stage_label'] = full.apply(assign_stage_label, axis=1)

    return full, encoder, scaler, feats

def id_final_label(df_rows) -> pd.Series:
    """ID-level final label from row CWD_status."""
    lab = (df_rows['CWD_status'].astype(str).str.lower() == 'positive').astype(int)
    return lab.groupby(df_rows['id']).max()

def stratified_group_folds(ids, id_labels, k=5, seed=42):
    """
    Simple stratified group split: partition pos IDs and neg IDs separately into k bins,
    then combine bins into folds. Keeps IDs disjoint and class balance roughly even.
    """
    rng = np.random.default_rng(seed)
    ids = np.asarray(ids)
    id_labels = pd.Series(id_labels).reindex(ids).astype(int)

    pos_ids = ids[id_labels.reindex(ids).to_numpy() == 1]
    neg_ids = ids[id_labels.reindex(ids).to_numpy() == 0]

    rng.shuffle(pos_ids)
    rng.shuffle(neg_ids)

    # split into k chunks (as even as possible)
    def chunk(arr, k):
        n = len(arr)
        sizes = [(n // k) + (1 if i < (n % k) else 0) for i in range(k)]
        out, idx = [], 0
        for s in sizes:
            out.append(arr[idx:idx+s])
            idx += s
        return out

    pos_chunks = chunk(pos_ids, k)
    neg_chunks = chunk(neg_ids, k)

    folds = []
    for i in range(k):
        val_ids = np.concatenate([pos_chunks[i], neg_chunks[i]]) if len(pos_chunks[i]) + len(neg_chunks[i]) > 0 else np.array([], dtype=ids.dtype)
        train_ids = np.setdiff1d(ids, val_ids, assume_unique=False)
        folds.append((train_ids, val_ids))
    return folds

# ── LOAD RAW DATA ─────────────────────────────────────────────────────────────
df_train_raw = pd.read_csv(train_raw)
if "datetime" in df_train_raw.columns:
    df_train_raw["datetime"] = pd.to_datetime(df_train_raw["datetime"], errors="coerce")

# ── Global OHE categories to keep columns identical across folds ──────────────
# (safe: purely unsupervised; avoids per-fold column drift)
all_cats = []
for c in cat_feats:
    all_cats.append(sorted(df_train_raw[c].dropna().astype(str).unique().tolist()))
ohe_categories = all_cats  # list per categorical feature, same order as cat_feats

# ── Build ID-level labels for stratified group CV ─────────────────────────────
id_labels = id_final_label(df_train_raw)               # Series indexed by id
all_ids   = id_labels.index.to_numpy()

# Create folds (ID-disjoint, roughly stratified)
folds = stratified_group_folds(all_ids, id_labels, k=CV_K, seed=RANDOM_SEED)

# For reporting
print(f"✅ Prepared {CV_K}-fold ID-disjoint CV (stratified at ID level).")
pos_ids = (id_labels[id_labels == 1].index.tolist())
neg_ids = (id_labels[id_labels == 0].index.tolist())
print(f"   ID positives: {len(pos_ids)} | ID negatives: {len(neg_ids)}")

# ── PROCESS & SAVE EACH FOLD ──────────────────────────────────────────────────
for fold_idx, (train_ids, val_ids) in enumerate(folds, start=1):
    fold_dir = FOLDS_DIR / f"fold_{fold_idx}"
    fold_dir.mkdir(exist_ok=True, parents=True)

    # Split rows by ID
    df_tr = df_train_raw[df_train_raw["id"].isin(train_ids)].copy().reset_index(drop=True)
    df_va = df_train_raw[df_train_raw["id"].isin(val_ids)].copy().reset_index(drop=True)

    # Fit encoder/scaler on TRAIN ONLY (encoder uses fixed categories)
    df_tr_proc, enc, sca, feat_cols = preprocess(
        df_tr, encoder=None, scaler=None, fit=True, fixed_categories=ohe_categories, add_label_column=True
    )
    # Transform VAL with those artifacts
    df_va_proc, _, _, _ = preprocess(
        df_va, encoder=enc, scaler=sca, fit=False, fixed_categories=ohe_categories, add_label_column=True
    )

    # Save per-fold artifacts & data
    df_tr_proc.to_csv(fold_dir / "CV_train_processed_infection_30_Conditional_exp.csv", index=False)
    df_va_proc.to_csv(fold_dir / "CV_val_processed_infection_30_Conditional_exp.csv", index=False)
    joblib.dump(enc, fold_dir / "CV_encoder_infection_30_Conditional_exp.pkl")
    joblib.dump(sca, fold_dir / "CV_scaler_infection_30_Conditional_exp.pkl")
    joblib.dump(feat_cols, fold_dir / "CV_feature_columns.pkl")

    # Fold summary
    print(f"\n—— Fold {fold_idx}/{CV_K} ——")
    print(f"train IDs: {len(np.unique(df_tr_proc['id']))} | rows: {len(df_tr_proc):,}")
    print(f" val  IDs: {len(np.unique(df_va_proc['id']))} | rows: {len(df_va_proc):,}")
    # label counts by ID
    tr_id_lab = df_tr_proc.groupby("id")["CWD_label"].max()
    va_id_lab = df_va_proc.groupby("id")["CWD_label"].max()
    print(f"   train ID pos/neg = {int((tr_id_lab==1).sum())}/{int((tr_id_lab==0).sum())}")
    print(f"    val  ID pos/neg = {int((va_id_lab==1).sum())}/{int((va_id_lab==0).sum())}")

# ── OPTIONAL: Global artifacts for a final single training / test transform ───
# If you still want a single final encoder/scaler (e.g., to train on all train_raw,
# then transform test), you can fit here on ALL train_raw (no labels used).
fit_global = True
if fit_global:
    df_full = df_train_raw.drop(columns=to_drop, errors='ignore').dropna(subset=num_feats + cat_feats).copy()
    df_full['datetime'] = pd.to_datetime(df_full['datetime'], errors='coerce')
    df_full = df_full.sort_values(['id','datetime']).reset_index(drop=True)

    # Fit global encoder with the same fixed categories to freeze column layout
    enc_global = make_ohe(ohe_categories)
    enc_global.fit(df_full[cat_feats])
    sca_global = MinMaxScaler().fit(df_full[num_feats])

    joblib.dump(enc_global, global_enc_path)
    joblib.dump(sca_global, global_sca_path)
    print(f"\n✅ Saved global encoder/scaler for future single-run training:")
    print(f"   {global_enc_path.name}, {global_sca_path.name}")

    # Optional: transform test with global artifacts
    try:
        df_test_raw = pd.read_csv(test_raw)
        df_test_raw['datetime'] = pd.to_datetime(df_test_raw['datetime'], errors='coerce')
        # Use same preprocess function but with global artifacts
        test_proc, _, _, feat_cols_global = preprocess(
            df_test_raw, encoder=enc_global, scaler=sca_global, fit=False,
            fixed_categories=ohe_categories, add_label_column=True
        )
        out_test = base/"CV_test_processed_infection_30_Conditional_exp.csv"
        test_proc.to_csv(out_test, index=False)
        joblib.dump(feat_cols_global, base/"feature_columns_global.pkl")
        print(f"✅ Wrote test_processed → {out_test}")
    except Exception as e:
        print(f"(Skipped test transform: {e})")

print("\n✅ Cross-validation preprocessing complete.")
print(f"   Folds root: {FOLDS_DIR}")


In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# cAE TRAINING (K-fold) — ID-disjoint CV using fold artifacts from preprocessing
#  - Per-fold: load CV_train/CV_val, enforce feature order, train cAE
#  - Save per-fold model, checkpoint, history, metadata
#  - Diagnostics: row-level margin AUC/PR on validation
#  - Aggregate summary across folds
# ──────────────────────────────────────────────────────────────────────────────
import os, random
from pathlib import Path

import numpy as np
import pandas as pd
import joblib
import tensorflow as tf
from tensorflow.keras import layers, Model, regularizers
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# ===== Reproducibility =====
os.environ["PYTHONHASHSEED"] = "0"
SEED = 13
random.seed(SEED); np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass

# ===== Paths =====
base = Path("<LOCAL_PATH>")
FOLDS_DIR = base / "cv_folds_k5"

# ===== Hyperparameters (shared across folds) =====
batch_size    = 128
patience      = 5
learning_rate = 1e-5
l2_reg        = 1e-4         # set 0.0 to disable
LAMBDA_DEV    = .75         # scales MSE via α = 1 + λ·d
EPOCHS        = 100
SHUFFLE_BUF   = 10_000


# ===== Utils =====
def load_df(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    if "datetime" in df.columns:
        df["datetime"] = pd.to_datetime(df["datetime"], errors="coerce")
    return df

def split_X_y(df: pd.DataFrame, feature_cols):
    X = df[feature_cols].values.astype(np.float32)
    if "CWD_label" in df.columns:
        y = df["CWD_label"].astype(np.float32).values.reshape(-1, 1)
    elif "CWD_status" in df.columns:
        y = (df["CWD_status"].astype(str).str.lower() == "positive").astype(np.float32).values.reshape(-1, 1)
    else:
        raise ValueError("No label column found (expected 'CWD_label' or 'CWD_status').")
    return X, y

AUTOTUNE = tf.data.AUTOTUNE
def make_ds(X, y, training=False, shuffle_buf=SHUFFLE_BUF):
    ds = tf.data.Dataset.from_tensor_slices(((X, y), X))
    if training:
        ds = ds.shuffle(min(shuffle_buf, len(X)), seed=SEED, reshuffle_each_iteration=True)
    return ds.batch(batch_size, drop_remainder=False).cache().prefetch(AUTOTUNE)

def build_cae(input_dim, l2_alpha=l2_reg):
    reg = regularizers.l2(l2_alpha) if l2_alpha and l2_alpha > 0 else None
    # Inputs
    x_in  = layers.Input(shape=(input_dim,), name="features")
    y_in  = layers.Input(shape=(1,),         name="cwd_label")
    # Encoder
    x = layers.Concatenate()([x_in, y_in])
    x = layers.Dense(32, activation='relu', kernel_regularizer=reg)(x)
    x = layers.Dense(16, activation='relu', kernel_regularizer=reg)(x)
    z = layers.Dense(4, activation='relu', kernel_regularizer=reg, name="latent")(x)
    encoder = Model([x_in, y_in], z, name="encoder")
    # Decoder
    z_in = layers.Input(shape=(4,), name="latent_in")
    y2   = layers.Input(shape=(1,), name="cwd_label_in")
    d = layers.Concatenate()([z_in, y2])
    d = layers.Dense(16, activation='relu', kernel_regularizer=reg)(d)
    d = layers.Dense(32, activation='relu', kernel_regularizer=reg)(d)
    x_hat = layers.Dense(input_dim, activation='linear')(d)
    decoder = Model([z_in, y2], x_hat, name="decoder")
    # Full cAE
    z = encoder([x_in, y_in])
    recon = decoder([z, y_in])
    model = Model([x_in, y_in], recon, name="conditional_autoencoder")
    return model, encoder, decoder

def get_scaled_mse_loss(alpha):
    def loss_fn(y_true, y_pred):
        # mean over features; multiplied by α = 1 + λ·d
        mse = tf.reduce_mean(tf.square(y_true - y_pred), axis=-1)
        return alpha * mse
    return loss_fn

# Collect fold summaries here
summ_rows = []

# ===== Iterate folds =====
fold_dirs = sorted([p for p in FOLDS_DIR.glob("fold_*") if p.is_dir()],
                   key=lambda p: int(p.name.split("_")[-1]))
if not fold_dirs:
    raise FileNotFoundError(f"No fold_* dirs in {FOLDS_DIR}. Did you run the CV preprocessing?")

for fold_path in fold_dirs:
    print(f"\n====================  TRAINING {fold_path.name}  ====================")
    train_path = fold_path / "CV_train_processed_infection_30_Conditional_exp.csv"
    val_path   = fold_path / "CV_val_processed_infection_30_Conditional_exp.csv"
    feat_cols_path = fold_path / "CV_feature_columns.pkl"

    # Per-fold outputs
    model_path      = fold_path / "conditional_autoencoder_model.h5"
    ckpt_path       = fold_path / "conditional_autoencoder_ckpt.weights.h5"
    history_path    = fold_path / "training_history.npy"
    train_meta_path = fold_path / "cae_training_metadata.pkl"

    # Load data & frozen feature order (from preprocessing)
    df_train = load_df(train_path)
    df_val   = load_df(val_path)
    feat_cols = joblib.load(feat_cols_path)

    # Build X,y using the frozen order
    X_train, y_train = split_X_y(df_train, feat_cols)
    X_val,   y_val   = split_X_y(df_val,   feat_cols)

    # Datasets
    train_ds = make_ds(X_train, y_train, training=True)
    val_ds   = make_ds(X_val,   y_val,   training=False)

    # Model + loss
    input_dim  = X_train.shape[1]
    LOSS_SCALE = 1.0 + LAMBDA_DEV * input_dim   # α
    cae, enc, dec = build_cae(input_dim)
    cae.compile(optimizer=tf.keras.optimizers.Adam(learning_rate),
                loss=get_scaled_mse_loss(LOSS_SCALE))

    # Callbacks
    early_stop = EarlyStopping(monitor="val_loss", patience=patience, restore_best_weights=True)
    ckpt = ModelCheckpoint(filepath=str(ckpt_path), save_weights_only=True,
                           monitor="val_loss", save_best_only=True)

    # Train
    hist = cae.fit(
        train_ds,
        epochs=EPOCHS,
        validation_data=val_ds,
        callbacks=[early_stop, ckpt],
        verbose=1
    )

    # Save model + history
    cae.save(model_path)
    np.save(history_path, hist.history, allow_pickle=True)
    print(f"✅ Saved model → {model_path.name}")
    print(f"✅ Saved history → {history_path.name}")
    print(f"✅ Best weights checkpoint → {ckpt_path.name}")

    # Quick reconstruction diagnostics (train/val)
    loss_fn = get_scaled_mse_loss(LOSS_SCALE)
    recon_tr = cae.predict((X_train, y_train), batch_size=batch_size, verbose=0)
    recon_va = cae.predict((X_val,   y_val),   batch_size=batch_size, verbose=0)
    err_tr = loss_fn(X_train, recon_tr).numpy().astype(np.float64)
    err_va = loss_fn(X_val,   recon_va).numpy().astype(np.float64)

    def summarize(errors, labels, name):
        neg = errors[labels.flatten() == 0]
        pos = errors[labels.flatten() == 1]
        print(f"[{name}] errors: mean±sd (neg) = {neg.mean():.6g} ± {neg.std():.6g} | "
              f"(pos) = {pos.mean():.6g} ± {pos.std():.6g} | N={len(errors):,}")
        return float(neg.mean()), float(pos.mean())

    tr_neg_mean, tr_pos_mean = summarize(err_tr, y_train, "train")
    va_neg_mean, va_pos_mean = summarize(err_va, y_val,   "val")

    # Optional: margin diagnostics on validation (dual reconstruction)
    try:
        N = X_val.shape[0]
        zeros = np.zeros((N,1), dtype=np.float32)
        ones  = np.ones((N,1), dtype=np.float32)
        recon_as_neg = cae.predict((X_val, zeros), batch_size=batch_size, verbose=0)
        recon_as_pos = cae.predict((X_val, ones ), batch_size=batch_size, verbose=0)
        err_as_neg = loss_fn(X_val, recon_as_neg).numpy().astype(np.float64).flatten()
        err_as_pos = loss_fn(X_val, recon_as_pos).numpy().astype(np.float64).flatten()
        margin = err_as_neg - err_as_pos
        # Row-level AUC / PR-AUC
        from sklearn.metrics import roc_auc_score, average_precision_score
        y_true = y_val.flatten().astype(int)
        row_auc = roc_auc_score(y_true, margin) if len(np.unique(y_true)) > 1 else np.nan
        row_ap  = average_precision_score(y_true, margin) if len(np.unique(y_true)) > 1 else np.nan
        print(f"[val] Row AUC (margin): {row_auc:.4f} | PR-AUC: {row_ap:.4f}")
    except Exception as e:
        row_auc, row_ap = np.nan, np.nan
        print(f"(Skipped margin AUC on val: {e})")

    # Persist per-fold training metadata
    train_meta = {
        "feature_columns": feat_cols,
        "input_dim": int(input_dim),
        "lambda_dev": float(LAMBDA_DEV),
        "loss_scale": float(LOSS_SCALE),   # α
        "batch_size": int(batch_size),
        "learning_rate": float(learning_rate),
        "epochs_ran": int(len(hist.history.get("loss", []))),
        "early_stopped": bool("val_loss" in hist.history),
        "best_ckpt_path": str(ckpt_path),
        "model_path": str(model_path),
        "seed": SEED,
        "val_diag": {
            "row_auc_margin": float(row_auc) if row_auc == row_auc else None,
            "row_pr_auc_margin": float(row_ap) if row_ap == row_ap else None,
            "val_err_neg_mean": float(va_neg_mean),
            "val_err_pos_mean": float(va_pos_mean),
        }
    }
    joblib.dump(train_meta, train_meta_path)
    print(f"✅ Training metadata saved → {train_meta_path.name}")

    # Add to summary
    best_val = float(np.min(hist.history.get("val_loss", [np.nan])))
    summ_rows.append({
        "fold": fold_path.name,
        "n_train_rows": len(df_train),
        "n_val_rows": len(df_val),
        "n_train_ids": int(df_train["id"].nunique()),
        "n_val_ids": int(df_val["id"].nunique()),
        "train_err_neg_mean": tr_neg_mean,
        "train_err_pos_mean": tr_pos_mean,
        "val_err_neg_mean": va_neg_mean,
        "val_err_pos_mean": va_pos_mean,
        "val_loss_min": best_val,
        "val_row_auc_margin": row_auc,
        "val_row_pr_auc_margin": row_ap,
    })

    # Clear session to free memory before next fold
    tf.keras.backend.clear_session()

# ===== Save CV summary =====
summary_df = pd.DataFrame(summ_rows)
summary_csv = FOLDS_DIR / "cv_training_summary.csv"
summary_df.to_csv(summary_csv, index=False)
print(f"\n✅ CV training complete. Summary → {summary_csv}")
print(summary_df.to_string(index=False))


## Freezing Validation

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# cAE VALIDATION (K-fold, RAW only) — per-fold τ + one pooled RAW threshold
#  - Flip-aware per fold (higher = more CWD+)
#  - τ = right-tail on NEG IDs (stable-only if available) + small std nudge
#  - Persist min_rows_for_test gate
#  - Per-fold metrics + cross-fold summary
#  - Save a single pooled RAW τ (median of per-fold τ)
# ──────────────────────────────────────────────────────────────────────────────
import os
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
import tensorflow as tf
from tensorflow.keras.models import load_model
from sklearn.metrics import roc_auc_score, average_precision_score

# ===== Paths =====
base = Path("<LOCAL_PATH>")
FOLDS_DIR = base / "cv_folds_k5"   # must match your CV preprocessing/training

# ===== Strictness knobs =====
ID_FPR_TARGET      = 0.425     # right-tail on NEG IDs
NUDGE_STD_FRACTION = 0.325     # + 0.15 * std(neg_scores)
MIN_ROWS_FOR_CAL   = 1     # only neg IDs with ≥ rows to calibrate τ
MIN_ROWS_FOR_TEST  = 1      # test-time eligibility gate (frozen here)


# ===== Helpers =====
def load_df(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    if "datetime" in df.columns:
        df["datetime"] = pd.to_datetime(df["datetime"], errors="coerce")
    return df

def split_X(df: pd.DataFrame, feat_cols):
    return df[feat_cols].values.astype(np.float32)

def loss_scaled_mse(alpha):
    def _loss(y_true, y_pred):
        mse = tf.reduce_mean(tf.square(y_true - y_pred), axis=-1)
        return alpha * mse
    return _loss

def fold_name_num(p: Path):
    try:
        return int(p.name.split("_")[-1])
    except Exception:
        return 9999

# ===== Iterate folds =====
fold_dirs = sorted([p for p in FOLDS_DIR.glob("fold_*") if p.is_dir()], key=fold_name_num)
if not fold_dirs:
    raise FileNotFoundError(f"No fold_* dirs in {FOLDS_DIR}")

summary_rows = []

for fold_path in fold_dirs:
    print(f"\n====================  VALIDATING {fold_path.name}  ====================")
    val_csv        = fold_path / "CV_val_processed_infection_30_Conditional_exp.csv"
    feat_cols_path = fold_path / "CV_feature_columns.pkl"
    train_meta_path= fold_path / "cae_training_metadata.pkl"
    model_path     = fold_path / "conditional_autoencoder_model.h5"
    out_policy     = fold_path / "cae_id_rule_params.pkl"

    # Artifacts
    df_val    = load_df(val_csv)
    feat_cols = joblib.load(feat_cols_path)
    meta      = joblib.load(train_meta_path)
    if not model_path.exists():  # fallback to meta path if needed
        model_path = Path(meta["model_path"])

    # Load model + loss (α saved at train time)
    model      = load_model(model_path, compile=False)
    lambda_dev = float(meta.get("lambda_dev", 0.5))
    loss_scale = float(meta.get("loss_scale", 1.0 + lambda_dev*len(feat_cols)))
    loss_fn    = loss_scaled_mse(loss_scale)

    # Build val tensors
    X_val = split_X(df_val, feat_cols)
    ids   = df_val["id"].values
    if "CWD_label" not in df_val.columns:
        raise ValueError("CWD_label missing in CV VAL file.")
    y_row = df_val["CWD_label"].astype(int).values

    # Dual recon + margin
    N = X_val.shape[0]
    zeros = np.zeros((N,1), dtype=np.float32)
    ones  = np.ones((N,1), dtype=np.float32)
    recon_neg = model.predict((X_val, zeros), batch_size=256, verbose=0)
    recon_pos = model.predict((X_val, ones ), batch_size=256, verbose=0)

    err_neg = loss_fn(X_val, recon_neg).numpy().astype(np.float64).flatten()
    err_pos = loss_fn(X_val, recon_pos).numpy().astype(np.float64).flatten()
    margin  = err_neg - err_pos  # >0 ⇒ fits CWD+ manifold better

    rows = pd.DataFrame({"id": ids, "margin": margin, "y": y_row})
    g = rows.groupby("id", sort=False)
    id_mean_margin = g["margin"].mean()
    id_n_rows      = g.size()
    id_label       = g["y"].max().astype(int)

    # Flip-aware score (higher = "more CWD+")
    neg_mask = (id_label == 0)
    pos_mask = (id_label == 1)
    neg_mean = float(id_mean_margin[neg_mask].mean()) if neg_mask.any() else np.nan
    pos_mean = float(id_mean_margin[pos_mask].mean()) if pos_mask.any() else np.nan
    if np.isfinite(neg_mean) and np.isfinite(pos_mean) and (pos_mean < neg_mean):
        score_name = "minus_mean_margin"
        id_score = -id_mean_margin
    else:
        score_name = "mean_margin"
        id_score = id_mean_margin

    # Calibrate τ on NEG IDs (stable-only if available)
    neg_scores_all = id_score[neg_mask]
    neg_rows_all   = id_n_rows[neg_mask]
    stable_neg     = (neg_rows_all >= MIN_ROWS_FOR_CAL)
    cal_base       = neg_scores_all[stable_neg] if stable_neg.any() else neg_scores_all
    if len(cal_base) == 0:
        raise ValueError(f"{fold_path.name}: no negative IDs to calibrate τ.")

    # Base τ and small nudge
    tau_q   = float(np.quantile(cal_base.values, 1.0 - ID_FPR_TARGET))
    tau_std = float(cal_base.values.std(ddof=1)) if len(cal_base) > 1 else 0.0
    tau     = tau_q + NUDGE_STD_FRACTION * tau_std

    # Predictions — raw (no gate) and with min-rows gate
    id_pred_raw   = (id_score > tau).astype(int)
    eligible_gate = (id_n_rows >= MIN_ROWS_FOR_TEST)
    id_pred_gate  = ((id_score > tau) & eligible_gate).astype(int)

    # Metrics helper
    def metrics(pred, label):
        tp = int(((pred == 1) & (label == 1)).sum())
        fp = int(((pred == 1) & (label == 0)).sum())
        fn = int(((pred == 0) & (label == 1)).sum())
        tn = int(((pred == 0) & (label == 0)).sum())
        prec = tp/(tp+fp) if (tp+fp) else 0.0
        rec  = tp/(tp+fn) if (tp+fn) else 0.0
        fpr  = fp/(fp+tn) if (fp+tn) else 0.0
        return tp, fp, fn, tn, prec, rec, fpr

    tp, fp, fn, tn, prec, rec, fpr = metrics(id_pred_raw, id_label)
    tp_g, fp_g, fn_g, tn_g, prec_g, rec_g, fpr_g = metrics(id_pred_gate, id_label)

    # Threshold-free diagnostics on val
    auc = pr = np.nan
    if id_label.nunique() > 1:
        auc = roc_auc_score(id_label.values, id_score.values)
        pr  = average_precision_score(id_label.values, id_score.values)

    print(f"[{fold_path.name}] score='{score_name}'")
    print(f"  neg_mean={neg_mean:.6g}, pos_mean={pos_mean:.6g}")
    print(f"  τ_q={tau_q:.6g} (FPR target={ID_FPR_TARGET:.3f}, base={'stable negs' if stable_neg.any() else 'all negs'})")
    print(f"  τ  ={tau:.6g} (+{NUDGE_STD_FRACTION:.2f}·std={tau_std:.6g})")
    print(f"  VAL (raw)  → P={prec:.3f} R={rec:.3f} FPR={fpr:.3f} | TP={tp} FP={fp} FN={fn} TN={tn} | AUC={auc:.3f} PR-AUC={pr:.3f}")
    print(f"  VAL (gate) → P={prec_g:.3f} R={rec_g:.3f} FPR={fpr_g:.3f} | TP={tp_g} FP={fp_g} FN={fn_g} TN={tn_g} | min_rows_test={MIN_ROWS_FOR_TEST}")

    # Persist per-fold frozen policy (RAW only)
    frozen = {
        "decision_level": "id_mean_margin",
        "score_name": score_name,                 # "mean_margin" | "minus_mean_margin"
        "tau_id_score": float(tau),
        "id_fpr_target": float(ID_FPR_TARGET),
        "lambda_dev": float(lambda_dev),
        "input_dim": int(meta["input_dim"]),
        "loss_scale": float(loss_scale),
        "model_path": str(model_path),
        "feature_columns": feat_cols,
        "calibrated_on": "stable_negatives" if stable_neg.any() else "all_negatives",
        "min_rows_for_cal": int(MIN_ROWS_FOR_CAL),
        "min_rows_for_test": int(MIN_ROWS_FOR_TEST),
        "fold": fold_path.name,
    }
    joblib.dump(frozen, out_policy)
    print(f"✅ Saved policy → {out_policy.name}")

    # Add to CV summary
    summary_rows.append({
        "fold": fold_path.name,
        "score_name": score_name,
        "tau": tau,
        "tau_quantile": tau_q,
        "tau_std": tau_std,
        "val_ids_pos": int((id_label==1).sum()),
        "val_ids_neg": int((id_label==0).sum()),
        "val_prec_raw": prec, "val_rec_raw": rec, "val_fpr_raw": fpr,
        "val_prec_gate": prec_g, "val_rec_gate": rec_g, "val_fpr_gate": fpr_g,
        "val_auc": float(auc) if auc==auc else np.nan,
        "val_pr_auc": float(pr) if pr==pr else np.nan,
    })

# ===== Save cross-fold summary =====
cv_summary = pd.DataFrame(summary_rows)
cv_summary_path = FOLDS_DIR / "cv_validation_summary.csv"
cv_summary.to_csv(cv_summary_path, index=False)
print(f"\n✅ CV validation complete. Summary → {cv_summary_path}")
print(cv_summary.to_string(index=False))

# ===== Pooled RAW τ for ensemble test-time use (single option) =====
try:
    taus = cv_summary["tau"].astype(float).to_numpy()
    tau_pooled_raw = float(np.median(taus))  # robust

    pooled = {
        "decision_level": "id_mean_margin_cv_pooled_RAW",
        "pooled_tau_raw": float(tau_pooled_raw),
        "tau_agg":        "median",
        "taus_per_fold":  taus.tolist(),
        "score_convention": "higher_is_CWD_plus_flip_aware",
        "min_rows_for_test": int(MIN_ROWS_FOR_TEST),
        "id_fpr_target": float(ID_FPR_TARGET),
        "nudge_std_fraction": float(NUDGE_STD_FRACTION),
        "min_rows_for_cal": int(MIN_ROWS_FOR_CAL),
        "fold_count": int(len(cv_summary)),
    }
    pooled_path = FOLDS_DIR / "cv_pooled_policy_RAW.pkl"
    joblib.dump(pooled, pooled_path)
    print(f"\n✅ Saved pooled RAW policy → {pooled_path}")
    print(f"   pooled τ (RAW) = {tau_pooled_raw:.6g} (median across folds)")
except Exception as e:
    print(f"\n⚠️  Skipped pooled RAW policy: {e}")


In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# cAE TESTING (K-fold, RAW only) — mean flip-aware score vs pooled τ
#  - Per fold: compute flip-aware ID score + aux (mean(err_pos), frac(margin>0))
#  - Ensemble: mean(score_f*), mean(err_pos_f*), mean(frac_pos_f*)
#  - Decision: score_mean > pooled_tau_raw with eligibility gate (eligible in ≥1 fold)
#  - Reviewer-friendly metrics on eligible set + CSV export
# ──────────────────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import joblib
from pathlib import Path
import tensorflow as tf
from tensorflow.keras.models import load_model
from sklearn.metrics import (
    confusion_matrix, precision_recall_fscore_support
)

# ===== Paths =====
base = Path("<LOCAL_PATH>")
FOLDS_DIR = base / "cv_folds_k5"   # must match your CV preprocessing/training/validation
test_csv  = base / "CV_test_processed_infection_30_Conditional_exp.csv"

# Output
ids_out_csv = base / "test_ids_cae_meanmargin_cv_RAW.csv"

# ===== Utilities =====
def loss_scaled_mse(alpha):
    def _loss(y_true, y_pred):
        mse = tf.reduce_mean(tf.square(y_true - y_pred), axis=-1)
        return alpha * mse
    return _loss

def fold_num(p: Path):
    try:
        return int(p.name.split("_")[-1])
    except Exception:
        return 9999

# ===== Load pooled RAW policy (from CV validation) =====
pooled_policy_path = FOLDS_DIR / "cv_pooled_policy_RAW.pkl"
if not pooled_policy_path.exists():
    raise FileNotFoundError(f"Missing pooled RAW policy: {pooled_policy_path} (run RAW-only CV validation)")
pooled = joblib.load(pooled_policy_path)
pooled_tau_raw = float(pooled["pooled_tau_raw"])
min_rows_test  = int(pooled.get("min_rows_for_test", 0))
print(f"Pooled RAW policy → τ={pooled_tau_raw:.6g} | min_rows_test={min_rows_test}")

# ===== Load test data =====
df = pd.read_csv(test_csv)
if "datetime" in df.columns:
    df["datetime"] = pd.to_datetime(df["datetime"], errors="coerce")
ids = df["id"].values
y_gt_rows = df["CWD_label"].astype(int).values if "CWD_label" in df.columns else None

# ===== Iterate folds =====
fold_dirs = sorted([p for p in FOLDS_DIR.glob("fold_*") if p.is_dir()], key=fold_num)
if not fold_dirs:
    raise FileNotFoundError(f"No fold_* dirs in {FOLDS_DIR}")

per_fold = []  # per-fold ID tables

for fidx, fold_path in enumerate(fold_dirs, start=1):
    print(f"\n====================  TESTING {fold_path.name}  ====================")
    # Artifacts
    feat_cols_path = fold_path / "CV_feature_columns.pkl"
    policy_path    = fold_path / "cae_id_rule_params.pkl"
    meta_path      = fold_path / "cae_training_metadata.pkl"
    model_path     = fold_path / "conditional_autoencoder_model.h5"

    feat_cols = joblib.load(feat_cols_path)
    policy    = joblib.load(policy_path)
    meta      = joblib.load(meta_path)
    if not model_path.exists():
        model_path = Path(meta["model_path"])

    # Feature check
    missing = [c for c in feat_cols if c not in df.columns]
    if missing:
        raise KeyError(f"{fold_path.name}: test CSV missing required features: {missing[:10]}{' ...' if len(missing)>10 else ''}")
    X = df[feat_cols].values.astype(np.float32)

    # Model + loss
    model = load_model(model_path, compile=False)
    lambda_dev = float(policy.get("lambda_dev", meta.get("lambda_dev", 0.5)))
    loss_scale = float(policy.get("loss_scale", meta.get("loss_scale", 1.0 + lambda_dev * len(feat_cols))))
    loss_fn    = loss_scaled_mse(loss_scale)

    # Frozen policy bits
    score_name    = policy.get("score_name", "mean_margin")
    tau_fold      = float(policy["tau_id_score"])
    min_rows_fold = int(policy.get("min_rows_for_test", min_rows_test))

    print(f"Policy({fold_path.name}): score={score_name} | τ_fold={tau_fold:.6g} | min_rows_test={min_rows_fold}")
    print(f"Model: {model_path.name} | α={loss_scale:.6g}")

    # Dual recon + row margin
    N = X.shape[0]
    zeros = np.zeros((N,1), dtype=np.float32)
    ones  = np.ones((N,1), dtype=np.float32)

    recon_neg = model.predict((X, zeros), batch_size=256, verbose=0)
    recon_pos = model.predict((X, ones ), batch_size=256, verbose=0)

    err_neg = loss_fn(X, recon_neg).numpy().astype(np.float64).flatten()
    err_pos = loss_fn(X, recon_pos).numpy().astype(np.float64).flatten()
    margin  = err_neg - err_pos  # >0 ⇒ fits CWD+ manifold better

    # ID aggregation
    rows = pd.DataFrame({"id": ids, "margin": margin, "err_as_pos": err_pos})
    g = rows.groupby("id", sort=False)
    id_mean_margin  = g["margin"].mean()
    id_mean_err_pos = g["err_as_pos"].mean()
    id_frac_pos     = g["margin"].apply(lambda s: (s > 0).mean())
    id_n_rows       = g.size()

    # Flip-aware score
    score = -id_mean_margin if score_name == "minus_mean_margin" else id_mean_margin

    # Eligibility gate
    eligible = (id_n_rows.values >= min_rows_fold) if min_rows_fold > 0 else np.ones(len(id_n_rows), dtype=bool)

    out = pd.DataFrame({
        "id": id_mean_margin.index,
        f"score_f{fidx}": score.values,
        f"mean_err_pos_f{fidx}": id_mean_err_pos.values,
        f"frac_pos_f{fidx}": id_frac_pos.values,
        f"eligible_f{fidx}": eligible.astype(int),
        f"n_rows_f{fidx}": id_n_rows.values,
        f"tau_f{fidx}": tau_fold,
    })
    per_fold.append(out)

# ===== Merge per-fold tables =====
ids_df = per_fold[0]
for t in per_fold[1:]:
    ids_df = ids_df.merge(t, on="id", how="outer")

# Attach ID GT (if present)
if y_gt_rows is not None:
    id_true = (
        pd.DataFrame({"id": ids, "y": y_gt_rows})
        .groupby("id", sort=False)["y"].max().astype(int)
        .reindex(ids_df["id"]).to_numpy()
    )
    ids_df["true_label_id"] = id_true

# ===== RAW-only ensemble aggregation =====
score_cols   = [c for c in ids_df.columns if c.startswith("score_f")]
errpos_cols  = [c for c in ids_df.columns if c.startswith("mean_err_pos_f")]
frac_cols    = [c for c in ids_df.columns if c.startswith("frac_pos_f")]
elig_cols    = [c for c in ids_df.columns if c.startswith("eligible_f")]

ids_df["score_mean"]   = ids_df[score_cols].mean(axis=1, skipna=True)
ids_df["err_pos_mean"] = ids_df[errpos_cols].mean(axis=1, skipna=True)
ids_df["frac_pos_mean"] = ids_df[frac_cols].mean(axis=1, skipna=True)

# Eligible in ≥1 fold
ids_df["eligible_any"] = (ids_df[elig_cols].fillna(0).astype(int).sum(axis=1) > 0).astype(int)
ids_df["eligible"]     = ids_df["eligible_any"]

# Decision: pooled RAW τ + eligibility
ids_df["id_pred"] = ((ids_df["score_mean"] > pooled_tau_raw) & (ids_df["eligible"] == 1)).astype(int)

# Save aggregated predictions
ids_df.to_csv(ids_out_csv, index=False)
print(f"\n✅ Wrote aggregated test IDs → {ids_out_csv}")

# ===== Table metrics (eligible set only) =====
def wilson_ci(k, n, z=1.96):
    if n == 0: return (np.nan, np.nan)
    p = k / n
    denom = 1 + z*z/n
    center = (p + z*z/(2*n)) / denom
    half = z * np.sqrt((p*(1-p)/n) + (z*z/(4*n*n))) / denom
    return (center - half, center + half)

if "true_label_id" in ids_df.columns:
    eval_df = ids_df[ids_df["eligible"] == 1].copy()
    y_true  = eval_df["true_label_id"].astype(int).to_numpy()
    y_pred  = eval_df["id_pred"].astype(int).to_numpy()

    TN, FP, FN, TP = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    prec, rec, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="binary", zero_division=0
    )
    spec = TN / (TN + FP) if (TN + FP) else np.nan
    sens = TP / (TP + FN) if (TP + FN) else np.nan
    bal_acc = 0.5 * (sens + spec) if np.isfinite(sens) and np.isfinite(spec) else np.nan
    tss = sens + spec - 1 if np.isfinite(sens) and np.isfinite(spec) else np.nan
    acc = (TP + TN) / (TP + TN + FP + FN) if (TP + TN + FP + FN) else np.nan
    fpr = FP / (FP + TN) if (FP + TN) else np.nan
    fnr = FN / (FN + TP) if (FN + TP) else np.nan
    prevalence = (TP + FN) / (TP + TN + FP + FN) if (TP + TN + FP + FN) else np.nan

    # 95% CIs (Wilson)
    prec_lo, prec_hi = wilson_ci(TP, TP + FP)
    rec_lo,  rec_hi  = wilson_ci(TP, TP + FN)
    spec_lo, spec_hi = wilson_ci(TN, TN + FP)
    tss_lo, tss_hi   = rec_lo + spec_lo - 1, rec_hi + spec_hi - 1

    table = pd.DataFrame({
        "N_eval": [int(len(eval_df))],
        "TP": [int(TP)], "FP": [int(FP)], "FN": [int(FN)], "TN": [int(TN)],
        "prevalence": [prevalence],
        "precision": [prec], "precision_CI_lo": [prec_lo], "precision_CI_hi": [prec_hi],
        "recall": [rec], "recall_CI_lo": [rec_lo], "recall_CI_hi": [rec_hi],
        "specificity": [spec], "specificity_CI_lo": [spec_lo], "specificity_CI_hi": [spec_hi],
        "sensitivity": [sens],
        "TSS": [tss], "TSS_CI_lo": [tss_lo], "TSS_CI_hi": [tss_hi],
        "F1": [f1],
        "accuracy": [acc],
        "balanced_accuracy": [bal_acc],
        "FPR": [fpr], "FNR": [fnr],
        "tau_pooled_raw": [pooled_tau_raw],   # fixed provenance
        "min_rows_test": [min_rows_test],
    })

    metrics_csv = ids_out_csv.with_name("test_metrics_table_RAW.csv")
    table.round(3).to_csv(metrics_csv, index=False)
    print(f"\n✅ Wrote metrics table → {metrics_csv}")
    print(table.to_string(index=False, float_format=lambda x: f'{x:.3f}'))

    suppressed = ids_df.loc[(ids_df["eligible"] == 0) & (ids_df["score_mean"] > pooled_tau_raw), "id"].tolist()
    if suppressed:
        print(f"(Suppressed by min_rows_test={min_rows_test}: {suppressed})")

    flagged = ids_df.loc[(ids_df["id_pred"] == 1) & (ids_df["eligible"] == 1), "id"].tolist()
    if flagged:
        print(f"\n🧠 IDs deemed CWD+ under mean-margin policy + min-rows gate:")
        print("[" + ", ".join(str(x) for x in flagged) + "]")
        print(f"Total flagged IDs: {len(flagged)} / {len(ids_df)}")


In [ ]:
# Define the actual CWD instances and unaffected CWD instances
actual_cwd_instances = [16, 18, 19, 21, 31, 53, 58, 72, 92, 102]
unaffected_cwd_instances = [3, 4, 7, 34, 38, 42, 43, 55, 56, 60, 77, 78, 82, 89, 95, 100, 111, 115, 119, 120, 129]


# You already have high_anomaly_ids from Step 1
print("High anomaly IDs:", flagged)

# Convert high_anomaly_ids to a set for easier comparison
high_deviation_ids = set(flagged)

# Calculate True Positives (TP), False Negatives (FN), False Positives (FP), and True Negatives (TN)
TP = len([ID for ID in actual_cwd_instances if ID in flagged])
FN = len([ID for ID in actual_cwd_instances if ID not in flagged])
FP = len([ID for ID in unaffected_cwd_instances if ID in flagged])
TN = len([ID for ID in unaffected_cwd_instances if ID not in flagged])

# Calculate False Positive Rate (FPR), False Negative Rate (FNR), and Accuracy
FPR = FP / (FP + TN) if (FP + TN) > 0 else 0
FNR = FN / (FN + TP) if (FN + TP) > 0 else 0
accuracy = (TP + TN) / (TP + TN + FP + FN) if (TP + TN + FP + FN) > 0 else 0

# Print the results
print(f"True Positives (TP): {TP}")
print(f"False Negatives (FN): {FN}")
print(f"False Positives (FP): {FP}")
print(f"True Negatives (TN): {TN}")
print(f"False Positive Rate (FPR): {FPR:.2f}")
print(f"False Negative Rate (FNR): {FNR:.2f}")
print(f"Accuracy: {accuracy:.2f}")


In [ ]:
# cae_overfit_diagnostics.py
import numpy as np, pandas as pd, joblib
from pathlib import Path
import tensorflow as tf
from tensorflow.keras.models import load_model
from sklearn.metrics import roc_auc_score, average_precision_score

BASE = Path("<LOCAL_PATH>")
FOLDS_DIR = BASE / "cv_folds_k5"

def load_df(p):
    df = pd.read_csv(p)
    if "datetime" in df.columns:
        df["datetime"] = pd.to_datetime(df["datetime"], errors="coerce")
    return df

def scaled_mse(alpha):
    def _loss(y_true, y_pred):
        mse = tf.reduce_mean(tf.square(y_true - y_pred), axis=-1)
        return alpha * mse
    return _loss

def fold_num(p: Path):
    try: return int(p.name.split("_")[-1])
    except: return 9999

rows = []
fold_dirs = sorted([p for p in FOLDS_DIR.glob("fold_*") if p.is_dir()], key=fold_num)
for fold in fold_dirs:
    tr_csv = fold / "CV_train_processed_infection_30_Conditional_exp.csv"
    va_csv = fold / "CV_val_processed_infection_30_Conditional_exp.csv"
    feats_p = fold / "CV_feature_columns.pkl"
    meta_p  = fold / "cae_training_metadata.pkl"
    model_p = fold / "conditional_autoencoder_model.h5"
    policy_p= fold / "cae_id_rule_params.pkl"

    df_tr = load_df(tr_csv); df_va = load_df(va_csv)
    feats = joblib.load(feats_p)
    meta  = joblib.load(meta_p)
    policy= joblib.load(policy_p)
    if not model_p.exists(): model_p = Path(meta["model_path"])
    model = load_model(model_p, compile=False)

    lam = float(policy.get("lambda_dev", meta.get("lambda_dev", 0.5)))
    alpha = float(policy.get("loss_scale", meta.get("loss_scale", 1.0 + lam*len(feats))))
    loss = scaled_mse(alpha)

    def eval_split(df, name):
        X = df[feats].values.astype(np.float32)
        y = df["CWD_label"].astype(int).values
        N = X.shape[0]
        z = np.zeros((N,1), dtype=np.float32); o = np.ones((N,1), dtype=np.float32)

        # recon under y=0 and y=1
        recon0 = model.predict((X, z), batch_size=256, verbose=0)
        recon1 = model.predict((X, o), batch_size=256, verbose=0)
        err0 = loss(X, recon0).numpy().astype(np.float64).flatten()
        err1 = loss(X, recon1).numpy().astype(np.float64).flatten()
        margin = err0 - err1  # >0 => fits CWD+ manifold better

        # gen gap (by class) using true label’s recon error
        true_recon = np.where(y==1, err1, err0)

        # label sensitivity at inference (flip y)
        # how much does error change if we condition on the *other* label?
        flipped_err = np.where(y==1, err0, err1)
        delta_flip = np.abs(true_recon - flipped_err)

        # row-level separation
        auc = roc_auc_score(y, margin) if len(np.unique(y))>1 else np.nan
        ap  = average_precision_score(y, margin) if len(np.unique(y))>1 else np.nan

        # summarize
        neg = true_recon[y==0]; pos = true_recon[y==1]
        return dict(
            split=name,
            n_rows=int(len(df)),
            n_ids=int(df["id"].nunique()),
            recon_neg_mean=float(np.mean(neg)) if len(neg) else np.nan,
            recon_pos_mean=float(np.mean(pos)) if len(pos) else np.nan,
            recon_gap=float((np.mean(pos)-np.mean(neg)) if (len(pos) and len(neg)) else np.nan),
            auc=float(auc) if auc==auc else np.nan,
            ap=float(ap) if ap==ap else np.nan,
            label_sensitivity_med=float(np.median(delta_flip)),
            label_sensitivity_p90=float(np.quantile(delta_flip, 0.90)),
        ), margin, y

    tr_stats, tr_margin, tr_y = eval_split(df_tr, "train")
    va_stats, va_margin, va_y = eval_split(df_va, "val")

    # τ stress test on VAL: recompute τ at FPR targets
    def tau_at_fpr(scores, labels, fpr_target):
        neg_scores = scores[labels==0]
        if len(neg_scores)==0: return np.nan
        q = 1.0 - fpr_target
        return float(np.quantile(neg_scores, q))

    def metrics_at_tau(scores, labels, tau):
        pred = (scores > tau).astype(int)
        tp = int(((pred==1)&(labels==1)).sum()); fp = int(((pred==1)&(labels==0)).sum())
        fn = int(((pred==0)&(labels==1)).sum()); tn = int(((pred==0)&(labels==0)).sum())
        prec = tp/(tp+fp) if (tp+fp) else 0.0
        rec  = tp/(tp+fn) if (tp+fn) else 0.0
        fpr  = fp/(fp+tn) if (fp+tn) else 0.0
        return prec, rec, fpr, tp, fp, fn, tn

    stress_rows = []
    for fpr_t in (0.05, 0.10, 0.20, 0.30, 0.45):
        tau = tau_at_fpr(va_margin, va_y, fpr_t)
        p,r,fpr, tp,fp,fn,tn = metrics_at_tau(va_margin, va_y, tau) if tau==tau else (np.nan,)*7
        stress_rows.append((fpr_t, tau, p, r, fpr, tp, fp, fn, tn))

    rows.append({
        "fold": fold.name, **tr_stats, **{f"val_{k}": v for k,v in va_stats.items() if k!="split"},
        "val_stress": stress_rows,
    })

# Print concise table
for r in rows:
    print(f"\n=== {r['fold']} ===")
    print(f"Train: n_ids={r['n_ids']}  recon_gap(pos-neg)={r['recon_gap']:.6g}  AUC={r['auc']:.3f}  AP={r['ap']:.3f}  label_sens_med={r['label_sensitivity_med']:.3g}")
    print(f" Val : n_ids={r['val_n_ids']} recon_gap(pos-neg)={r['val_recon_gap']:.6g} AUC={r['val_auc']:.3f} AP={r['val_ap']:.3f} label_sens_med={r['val_label_sensitivity_med']:.3g}")
    print(" Val τ stress (target_FPR → τ, precision, recall, realized FPR):")
    for fpr_t, tau, p, rec, fpr, tp, fp, fn, tn in r["val_stress"]:
        print(f"  FPR@{int(100*fpr_t):>2d}% → τ={tau:.6g} | P={p:.3f} R={rec:.3f} FPR={fpr:.3f} | TP={tp} FP={fp} FN={fn} TN={tn}")


In [ ]:
# =================== CV-AWARE BINNED PLOTS (duration in days) ==================
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

# ---------- 0) Read timestamps & build duration bins (quantiles on DAYS) ------
df_full = pd.read_csv(test_csv if "test_csv" in globals() else test_csv)
time_col = "date_time" if "date_time" in df_full.columns else ("datetime" if "datetime" in df_full.columns else None)
if time_col is None:
    raise KeyError("No 'date_time' or 'datetime' column in test CSV.")
df_full[time_col] = pd.to_datetime(df_full[time_col], errors="coerce")

id_start = df_full.groupby("id")[time_col].min()
id_end   = df_full.groupby("id")[time_col].max()
id_duration_days = (id_end - id_start).dt.total_seconds() / 86400.0

# Limit to IDs present in ids_df
id_duration_days = id_duration_days.reindex(ids_df["id"].unique())

# Quantile bins (equal-count) with robustness for ties/small N
N_BINS = 7
q = min(N_BINS, max(1, id_duration_days.dropna().nunique()))
cats, edges = pd.qcut(id_duration_days, q=q, retbins=True, duplicates="drop")

def _label(left, right, i=None):
    L = int(np.floor(left))
    R = int(np.ceil(right))
    base = f"{L}–{R} days" if R > L else f"{L} days"
    if i is not None:
        base += f" (bin {i})"
    return base

intervals = cats.cat.categories
bin_labels = [_label(iv.left, iv.right, i) for i, iv in enumerate(intervals)]
cats = cats.cat.rename_categories(bin_labels)

# Map each ID to a bin label and attach to ids_df
id_to_bin = pd.Series(cats.values, index=id_duration_days.index).to_dict()
ids_df["bin"] = ids_df["id"].map(id_to_bin)

# Preserve categorical order
bin_order = bin_labels

# ---------- 1) Helpers: metrics & CIs ----------------------------------------
def wilson_ci(k, n, z=1.96):
    if n == 0: return (np.nan, np.nan)
    p = k / n
    denom = 1 + z*z/n
    center = (p + z*z/(2*n)) / denom
    half = z * np.sqrt((p*(1-p)/n) + (z*z/(4*n*n))) / denom
    return (center - half, center + half)

def counts_to_metrics(TN, FP, FN, TP):
    N = TP + TN + FP + FN
    acc  = (TP + TN) / N if N else np.nan
    prec = TP / (TP + FP) if (TP + FP) else np.nan
    sens = TP / (TP + FN) if (TP + FN) else np.nan
    spec = TN / (TN + FP) if (TN + FP) else np.nan
    tss  = sens + spec - 1 if (np.isfinite(sens) and np.isfinite(spec)) else np.nan
    # CIs
    acc_lo, acc_hi   = wilson_ci(TP + TN, N) if N else (np.nan, np.nan)
    prec_lo, prec_hi = wilson_ci(TP, TP + FP) if (TP + FP) else (np.nan, np.nan)
    sens_lo, sens_hi = wilson_ci(TP, TP + FN) if (TP + FN) else (np.nan, np.nan)
    spec_lo, spec_hi = wilson_ci(TN, TN + FP) if (TN + FP) else (np.nan, np.nan)
    tss_lo, tss_hi   = (sens_lo + spec_lo - 1, sens_hi + spec_hi - 1) if (np.isfinite(sens_lo) and np.isfinite(spec_lo)) else (np.nan, np.nan)
    return dict(accuracy=acc, accuracy_lo=acc_lo, accuracy_hi=acc_hi,
                precision=prec, precision_lo=prec_lo, precision_hi=prec_hi,
                tss=tss, tss_lo=tss_lo, tss_hi=tss_hi)

def t_crit_95(n):
    # two-sided 95% t critical (approx) for small n
    # n = number of observations (folds) used in mean; df=n-1
    # conservative mapping for common n; fallback to 1.96
    table = {2: 12.706, 3: 4.303, 4: 3.182, 5: 2.776, 6: 2.571, 7: 2.447,
             8: 2.365, 9: 2.306, 10: 2.262, 12: 2.201, 15: 2.131, 20: 2.086, 30: 2.042}
    if n in table: return table[n]
    if n < 2: return np.nan
    if n >= 30: return 2.042
    return 2.262  # ~n=10 default

# ---------- 2A) ENSEMBLE (pooled) per-bin metrics using ids_df["id_pred"] ----
def build_bin_table_ensemble(ids_df, bin_order, pos_label_col="true_label_id"):
    rows = []
    for b in bin_order:
        sub = ids_df[(ids_df["bin"] == b) & (ids_df["eligible"] == 1)]
        if sub.empty:
            rows.append({"bin": b, "n_eval": 0, "TP":0, "FP":0, "FN":0, "TN":0,
                         "accuracy": np.nan, "accuracy_lo": np.nan, "accuracy_hi": np.nan,
                         "precision": np.nan, "precision_lo": np.nan, "precision_hi": np.nan,
                         "tss": np.nan, "tss_lo": np.nan, "tss_hi": np.nan})
            continue
        if pos_label_col not in sub.columns:
            raise KeyError(f"'{pos_label_col}' not found in ids_df. Add true labels at ID level.")
        y_true = sub[pos_label_col].astype(int).to_numpy()
        y_pred = sub["id_pred"].astype(int).to_numpy()
        TN, FP, FN, TP = confusion_matrix(y_true, y_pred, labels=[0,1]).ravel()
        met = counts_to_metrics(TN, FP, FN, TP)
        rows.append({"bin": b, "n_eval": len(sub), "TP":TP, "FP":FP, "FN":FN, "TN":TN, **met})
    out = pd.DataFrame(rows)
    out["bin"] = pd.Categorical(out["bin"], categories=bin_order, ordered=True)
    return out.sort_values("bin")

# ---------- 2B) ACROSS-FOLD per-bin metrics (mean ±95% CI over folds) --------
def build_bin_table_across_folds(ids_df, bin_order, pos_label_col="true_label_id"):
    # Detect fold indices
    fold_idxs = sorted(int(re.search(r"score_f(\d+)", c).group(1)) for c in ids_df.columns if re.match(r"score_f\d+", c))
    if not fold_idxs:
        raise RuntimeError("No per-fold score columns found (score_f*).")

    rows = []
    for b in bin_order:
        # Per-fold metric vectors (variable length due to NaNs/eligibility)
        acc_list, prec_list, tss_list = [], [], []
        TP_sum = FP_sum = FN_sum = TN_sum = 0  # (optional) sum counts over folds for info
        n_eval_mean = []

        for f in fold_idxs:
            need = [f"score_f{f}", f"eligible_f{f}"]
            if any(col not in ids_df.columns for col in need):
                continue
            sub = ids_df.loc[(ids_df["bin"] == b) & (ids_df[f"eligible_f{f}"] == 1)].copy()
            if sub.empty or (pos_label_col not in sub.columns):
                continue

            # Decision per fold using that fold's τ
            tau_col = f"tau_f{f}"
            if tau_col not in sub.columns:
                # try to pull from whole ids_df if constant per fold
                if tau_col in ids_df.columns and ids_df[tau_col].notna().any():
                    tau_val = float(ids_df[tau_col].dropna().iloc[0])
                else:
                    raise KeyError(f"Missing '{tau_col}' for fold {f}.")
                sub["id_pred_f"] = (sub[f"score_f{f}"] > tau_val).astype(int)
            else:
                # Some exports store a constant τ per row; safe to use the column directly
                tau_vals = sub[tau_col]
                # If not constant per row, this still works element-wise
                sub["id_pred_f"] = (sub[f"score_f{f}"] > tau_vals).astype(int)

            y_true = sub[pos_label_col].astype(int).to_numpy()
            y_pred_f = sub["id_pred_f"].astype(int).to_numpy()

            TN, FP, FN, TP = confusion_matrix(y_true, y_pred_f, labels=[0,1]).ravel()
            met = counts_to_metrics(TN, FP, FN, TP)
            if np.isfinite(met["accuracy"]):  acc_list.append(met["accuracy"])
            if np.isfinite(met["precision"]): prec_list.append(met["precision"])
            if np.isfinite(met["tss"]):       tss_list.append(met["tss"])

            TP_sum += TP; FP_sum += FP; FN_sum += FN; TN_sum += TN
            n_eval_mean.append(len(sub))

        def agg(v):
            v = np.asarray(v, float)
            v = v[np.isfinite(v)]
            if v.size == 0: return (np.nan, np.nan, np.nan)
            m = float(np.mean(v))
            s = float(np.std(v, ddof=1)) if v.size > 1 else np.nan
            tcrit = t_crit_95(v.size)
            ci = tcrit * (s / np.sqrt(v.size)) if (np.isfinite(s) and v.size > 1) else np.nan
            return (m, m - ci if np.isfinite(ci) else np.nan, m + ci if np.isfinite(ci) else np.nan)

        acc_m, acc_lo, acc_hi   = agg(acc_list)
        prec_m, prec_lo, prec_hi= agg(prec_list)
        tss_m, tss_lo, tss_hi   = agg(tss_list)

        rows.append({
            "bin": b,
            "n_eval_mean": float(np.mean(n_eval_mean)) if n_eval_mean else 0.0,
            "TP": TP_sum, "FP": FP_sum, "FN": FN_sum, "TN": TN_sum,
            "accuracy": acc_m, "accuracy_lo": acc_lo, "accuracy_hi": acc_hi,
            "precision": prec_m, "precision_lo": prec_lo, "precision_hi": prec_hi,
            "tss": tss_m, "tss_lo": tss_lo, "tss_hi": tss_hi,
            "n_folds_used_acc": len(acc_list), "n_folds_used_prec": len(prec_list), "n_folds_used_tss": len(tss_list)
        })

    out = pd.DataFrame(rows)
    out["bin"] = pd.Categorical(out["bin"], categories=bin_order, ordered=True)
    return out.sort_values("bin")

# ---------- 3) Build both tables ------------------------------------------------
if "true_label_id" not in ids_df.columns:
    raise KeyError("ids_df is missing 'true_label_id'. Attach ground-truth labels at ID level before plotting.")

bin_df_ens  = build_bin_table_ensemble(ids_df, bin_order)        # pooled/ensemble
bin_df_fold = build_bin_table_across_folds(ids_df, bin_order)    # across-fold means

print("\nENSEMBLE per-bin counts:")
print(bin_df_ens[["bin","n_eval","TP","FP","FN","TN"]].to_string(index=False))
print("\nACROSS-FOLD per-bin folds-used:")
print(bin_df_fold[["bin","n_folds_used_acc","n_folds_used_prec","n_folds_used_tss"]].to_string(index=False))

# ---------- 4) Plot #1: ENSEMBLE (same style you had) --------------------------
def plot_bars_three(ddd: pd.DataFrame, title: str, fname: str):
    # Ensure there are rows
    if ddd is None or len(ddd) == 0:
        print("⚠️ Nothing to plot (empty DataFrame).")
        return

    # Build numeric x and string labels
    ddd = ddd.copy()
    labels = ddd["bin"].astype(str).fillna("(missing)").tolist()
    x = np.arange(len(labels), dtype=float)

    print(f"🧭 Plotting {len(labels)} bins:", labels)

    # Make sure columns exist even if all-NaN
    for col in ["accuracy","accuracy_lo","accuracy_hi",
                "precision","precision_lo","precision_hi",
                "tss","tss_lo","tss_hi","TP","FP","FN","TN"]:
        if col not in ddd.columns:
            ddd[col] = np.nan

    fig, ax = plt.subplots(2, 3, figsize=(18, 10), sharex=False)  # <- no sharex to avoid surprises

    # --- Top row: metrics ---
    panels = [
        ("accuracy","accuracy_lo","accuracy_hi","Accuracy", 0.0, 1.0, "#cfe1f2"),
        ("precision","precision_lo","precision_hi","Precision", 0.0, 1.0, "#cfe1f2"),
        ("tss","tss_lo","tss_hi","TSS", -1.0, 1.0, "#dcd3f2"),
    ]
    for i, (m, lo, hi, ttl, ymin, ymax, color) in enumerate(panels):
        center = ddd[m].to_numpy(float)
        lo_arr = center - ddd[lo].to_numpy(float)
        hi_arr = ddd[hi].to_numpy(float) - center

        # finite mask for errorbars; still draw placeholders for NaNs
        msk = np.isfinite(center) & np.isfinite(lo_arr) & np.isfinite(hi_arr)

        # placeholder bars so bins show even if metric is NaN
        ax[0, i].bar(x, np.nan_to_num(center, nan=0.0), color="#eeeeee", edgecolor="black",
                     linewidth=0.6, width=0.8, zorder=1)

        # overlay real bars only where finite
        ax[0, i].bar(x[msk], center[msk], color=color, edgecolor="black",
                     linewidth=0.6, width=0.8, zorder=2)

        # error bars
        if np.any(msk):
            ax[0, i].errorbar(x[msk], center[msk],
                              yerr=np.vstack([lo_arr[msk], hi_arr[msk]]),
                              fmt="none", ecolor="black", capsize=5, elinewidth=1.2, zorder=3)

        ax[0, i].set_title(ttl)
        ax[0, i].set_ylim(ymin, ymax)
        ax[0, i].set_xlim(-0.5, len(x)-0.5)  # <- force x-range to bins
        ax[0, i].grid(axis="y", linestyle="--", alpha=0.35)
        ax[0, i].set_ylabel("Score")

    # --- Bottom row: counts (TP, FP, FN) ---
    for j, cm_m, col in [(0,"TP","#f6c0c0"), (1,"FP","#f6c0c0"), (2,"FN","#f6c0c0")]:
        vals = ddd[cm_m].fillna(0).to_numpy(int)
        ax[1, j].bar(x, vals, color=col, edgecolor="black", linewidth=0.6)
        ax[1, j].set_title(cm_m)
        ax[1, j].set_ylabel("Count")
        ax[1, j].set_xlim(-0.5, len(x)-0.5)
        ax[1, j].grid(axis="y", linestyle="--", alpha=0.35)

    # Replace the 4th axis with TN
    fig.delaxes(ax[1, 2])
    tn_ax = fig.add_subplot(2, 3, 6)
    tn_vals = ddd["TN"].fillna(0).to_numpy(int)
    tn_ax.bar(x, tn_vals, color="#cfeccf", edgecolor="black", linewidth=0.6)
    tn_ax.set_title("TN")
    tn_ax.set_ylabel("Count")
    tn_ax.set_xlim(-0.5, len(x)-0.5)
    tn_ax.grid(axis="y", linestyle="--", alpha=0.35)

    # X ticks on the bottom row and TN axis
    for a in [ax[1,0], ax[1,1], tn_ax]:
        a.set_xticks(x)
        a.set_xticklabels(labels, rotation=30, ha="right")
    # Clear top-row tick labels
    for a in ax[0, :]:
        a.set_xticks(x)
        a.set_xticklabels([])

    plt.suptitle(title, fontsize=16, y=0.98)
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    out_png = base / f"{fname}.png"; out_pdf = base / f"{fname}.pdf"
    plt.savefig(out_png, bbox_inches="tight"); plt.savefig(out_pdf, bbox_inches="tight")
    print(f"📊 Saved: {out_png}\n📊 Saved: {out_pdf}")
    plt.show(); plt.close(fig)

plot_bars_three(bin_df_ens,  "30% Infection CV Ensemble: Metrics & Counts by Duration Bin", "cv_bins_ensemble")
plot_bars_three(bin_df_fold, "30% Infection CV Across-Fold (mean±95%CI): Metrics by Duration Bin", "cv_bins_acrossfold")


In [ ]:
# ===================== cAE (CV) Feature Contributions — manuscript-ready =====================
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict
import joblib
import tensorflow as tf
from tensorflow.keras.models import load_model

# --- define outdir ONCE, as a Path, BEFORE any use ---
outdir = Path("<LOCAL_PATH>")
outdir.mkdir(parents=True, exist_ok=True)


# ------------ Config / inputs (must exist from your CV run) ------------
# base, FOLDS_DIR, test_csv, ids_df, pooled_tau_raw already defined in your CV script

BATCH = 256

# --- Helper: scaled MSE row loss (matches your training/validation) ---
def loss_scaled_mse(alpha):
    def _loss(y_true, y_pred):
        mse = tf.reduce_mean(tf.square(y_true - y_pred), axis=-1)
        return alpha * mse
    return _loss

# ------------ Load test data ------------
df_test = pd.read_csv(test_csv)
if "datetime" in df_test.columns:
    df_test["datetime"] = pd.to_datetime(df_test["datetime"], errors="coerce")
ids_rows = df_test["id"].to_numpy()

# ------------ Walk folds, compute recon_pos/neg & per-row losses ------------
fold_dirs = sorted([p for p in Path(FOLDS_DIR).glob("fold_*") if p.is_dir()],
                   key=lambda p: int(p.name.split("_")[-1]))
if not fold_dirs:
    raise FileNotFoundError(f"No fold_* dirs in {FOLDS_DIR}")

residual_sq_list = []   # per-fold per-feature squared residuals under POS
err_pos_list     = []   # per-fold per-row loss under POS
err_neg_list     = []   # per-fold per-row loss under NEG

for fidx, fold_path in enumerate(fold_dirs, start=1):
    feat_cols_path = fold_path / "CV_feature_columns.pkl"
    policy_path    = fold_path / "cae_id_rule_params.pkl"
    meta_path      = fold_path / "cae_training_metadata.pkl"
    model_path     = fold_path / "conditional_autoencoder_model.h5"

    feat_cols = joblib.load(feat_cols_path)
    policy    = joblib.load(policy_path)
    meta      = joblib.load(meta_path)
    if not model_path.exists():
        model_path = Path(meta["model_path"])

    # Column alignment to this fold’s expected order
    missing = [c for c in feat_cols if c not in df_test.columns]
    if missing:
        raise KeyError(f"{fold_path.name}: test CSV missing features: {missing[:10]}{' ...' if len(missing)>10 else ''}")
    Xf = df_test[feat_cols].to_numpy(dtype=np.float32)

    # Load model & build loss
    model = load_model(model_path, compile=False)
    lambda_dev = float(policy.get("lambda_dev", meta.get("lambda_dev", 0.5)))
    loss_scale = float(policy.get("loss_scale", meta.get("loss_scale", 1.0 + lambda_dev * len(feat_cols))))
    loss_fn    = loss_scaled_mse(loss_scale)

    # Dual reconstruction (condition: 0=neg, 1=pos)
    N = Xf.shape[0]
    zeros = np.zeros((N,1), dtype=np.float32)
    ones  = np.ones((N,1), dtype=np.float32)

    recon_neg = model.predict((Xf, zeros), batch_size=BATCH, verbose=0)
    recon_pos = model.predict((Xf, ones ), batch_size=BATCH, verbose=0)

    # Per-row losses (scaled MSE, same as your decision metric components)
    err_neg = loss_fn(Xf, recon_neg).numpy().astype(np.float64).ravel()
    err_pos = loss_fn(Xf, recon_pos).numpy().astype(np.float64).ravel()

    # Per-feature squared residuals under POS (for contributions)
    rsq_pos = (Xf - recon_pos)**2  # shape: (N_rows, n_features)

    residual_sq_list.append(rsq_pos)
    err_pos_list.append(err_pos)
    err_neg_list.append(err_neg)

# ------------ Fold-averaged residuals & margins ------------
# (Averaging across folds keeps everything in the test space and avoids picking a single fold.)
residual_sq = np.nanmean(np.stack(residual_sq_list, axis=0), axis=0)  # (N, d)
err_pos_mean = np.nanmean(np.stack(err_pos_list, axis=0), axis=0)     # (N,)
err_neg_mean = np.nanmean(np.stack(err_neg_list, axis=0), axis=0)     # (N,)
margin_mean  = err_neg_mean - err_pos_mean                            # >0 ⇒ row looks more like CWD+

# Build a rows frame for masks
rows_cae = pd.DataFrame({
    "id": ids_rows,
    "err_pos_mean": err_pos_mean,
    "err_neg_mean": err_neg_mean,
    "margin_mean": margin_mean,
})
rows_cae["row_flag_cae"] = rows_cae["margin_mean"] > 0  # define “flagged” for cAE

# ------------ Build masks from CV decisions (ids_df) ------------
# Expect ids_df has: id, true_label_id, id_pred, eligible
required_cols = {"id", "true_label_id", "id_pred", "eligible"}
if not required_cols.issubset(set(ids_df.columns)):
    raise KeyError(f"ids_df must contain {required_cols}, found {sorted(ids_df.columns)}")

eval_ids = set(ids_df.loc[ids_df["eligible"] == 1, "id"])
TP_ids   = set(ids_df.query("eligible == 1 and true_label_id == 1 and id_pred == 1")["id"])
FP_ids   = set(ids_df.query("eligible == 1 and true_label_id == 0 and id_pred == 1")["id"])
FN_ids   = set(ids_df.query("eligible == 1 and true_label_id == 1 and id_pred == 0")["id"])
TN_ids   = set(ids_df.query("eligible == 1 and true_label_id == 0 and id_pred == 0")["id"])

rid = rows_cae["id"]
mask_all   = np.ones(len(rows_cae), dtype=bool)
mask_flag  = rows_cae["row_flag_cae"].to_numpy(bool)               # cAE “flagged” = margin>0
mask_eval  = rid.isin(eval_ids).to_numpy()
m_tp       = rid.isin(TP_ids).to_numpy()
m_fp       = rid.isin(FP_ids).to_numpy()
m_fn       = rid.isin(FN_ids).to_numpy()
m_tn       = rid.isin(TN_ids).to_numpy()

# ------------ Contribution helpers ------------
feat_names = list(df_test.columns.intersection(residual_sq.shape[1] and df_test.columns))  # for safety
# If fold feature sets are identical, use the feature list from the last loop
# (better: load once from a fold)
try:
    feat_names = list(feat_cols)
except NameError:
    feat_names = [f"f{i}" for i in range(residual_sq.shape[1])]

def contrib_pct(rsq, mask):
    """Mean per-feature residual (under POS), normalized to % across features."""
    if rsq.size == 0 or not np.any(mask):
        return np.full(rsq.shape[1], np.nan)
    m = np.nanmean(rsq[mask], axis=0)
    s = np.nansum(m)
    return (m / s * 100.0) if s > 0 else np.full_like(m, np.nan, dtype=float)

contrib_all  = contrib_pct(residual_sq, mask_all)
contrib_flag = contrib_pct(residual_sq, mask_flag)
contrib_eval = contrib_pct(residual_sq, mask_eval)

# Per-ID variability (erraticness)
id_to_rows = defaultdict(list)
for r_i, id_i in enumerate(rid):
    id_to_rows[id_i].append(r_i)

def per_id_contrib_stats(mask):
    per_id = []
    for i, idxs in id_to_rows.items():
        idxs = np.asarray(idxs, int)
        idxs = idxs[mask[idxs]]
        if idxs.size == 0:
            continue
        m = np.nanmean(residual_sq[idxs], axis=0)
        s = np.nansum(m)
        per_id.append((i, (m / s * 100.0) if s > 0 else np.full_like(m, np.nan)))
    if not per_id:
        return dict(median_feature_sd_pp=np.nan, n_ids=0)
    M = np.vstack([v for _, v in per_id])
    sd_pp = np.nanstd(M, axis=0)  # per-feature SD across IDs (percentage points)
    return dict(median_feature_sd_pp=float(np.nanmedian(sd_pp)), n_ids=len(per_id))

err_all  = per_id_contrib_stats(mask_all)
err_flag = per_id_contrib_stats(mask_flag)
err_eval = per_id_contrib_stats(mask_eval)

def topk_share_pct(pct, k=3):
    p = np.asarray(pct, float)
    p = p[np.isfinite(p)]
    if p.size == 0: return np.nan
    p_sorted = np.sort(p)[::-1]
    return float(np.sum(p_sorted[:min(k, p_sorted.size)]))

top3_all,  top5_all  = topk_share_pct(contrib_all, 3), topk_share_pct(contrib_all, 5)
top3_flag, top5_flag = topk_share_pct(contrib_flag, 3), topk_share_pct(contrib_flag, 5)
top3_eval, top5_eval = topk_share_pct(contrib_eval, 3), topk_share_pct(contrib_eval, 5)

# ------------ Tables: per-feature & summary ------------
K = 15
df_contrib = pd.DataFrame({
    "feature": feat_names,
    "All_%":      contrib_all,
    "Flagged_%":  contrib_flag,
    "Eval_%":     contrib_eval,
})
df_contrib["Delta_Flagged_minus_All_pp"] = df_contrib["Flagged_%"] - df_contrib["All_%"]
df_top = df_contrib.sort_values("Flagged_%", ascending=False).head(K)

print("\n=== cAE (CV) Feature Contributions — % of reconstruction error under POS ===")
print(df_top.to_string(index=False, float_format=lambda x: f"{x:.2f}"))

summary_tbl = pd.DataFrame([
    {"Group":"All rows",    "Top3_share_%": top3_all,  "Top5_share_%": top5_all,  "Median_feat_SD_pp": err_all["median_feature_sd_pp"],  "n_ids": err_all["n_ids"]},
    {"Group":"Flagged rows","Top3_share_%": top3_flag, "Top5_share_%": top5_flag, "Median_feat_SD_pp": err_flag["median_feature_sd_pp"], "n_ids": err_flag["n_ids"]},
    {"Group":"Eval IDs",    "Top3_share_%": top3_eval, "Top5_share_%": top5_eval, "Median_feat_SD_pp": err_eval["median_feature_sd_pp"], "n_ids": err_eval["n_ids"]},
])
summary_tbl["ΔFlagged−All_Top3_pp"] = summary_tbl["Top3_share_%"].iloc[1] - summary_tbl["Top3_share_%"].iloc[0]
summary_tbl["ΔFlagged−All_Top5_pp"] = summary_tbl["Top5_share_%"].iloc[1] - summary_tbl["Top5_share_%"].iloc[0]
summary_tbl["ΔFlagged−All_SD_pp"]   = summary_tbl["Median_feat_SD_pp"].iloc[1] - summary_tbl["Median_feat_SD_pp"].iloc[0]

print("\n=== Simple concentration & variability summary ===")
print(summary_tbl.to_string(index=False, float_format=lambda x: f"{x:.2f}"))

# ------------ Category roll-up (movement story) ------------
def categorize(fname: str) -> str:
    f = fname.lower()
    if "velo" in f: return "velocity"
    if "angle" in f: return "turning-angle"
    if "sinu" in f:  return "sinuosity"
    if "skew" in f or "kurt" in f: return "distributional-moment"
    if "rolling" in f or "mean" in f or "sd" in f: return "rolling-statistic"
    if f in {"low","est","high"} or "model" in f: return "model-derived"
    if "season" in f: return "season"
    return "other"

cat = pd.DataFrame({
    "feature": feat_names,
    "category": [categorize(f) for f in feat_names],
    "All_%": contrib_all, "Flagged_%": contrib_flag, "Eval_%": contrib_eval
})
cat_roll = (cat.groupby("category")[["All_%","Flagged_%","Eval_%"]]
              .sum().sort_values("Flagged_%", ascending=False))
cat_roll["Delta_Flagged_minus_All_pp"] = cat_roll["Flagged_%"] - cat_roll["All_%"]
print("\n=== Category roll-up (% of recon error) ===")
print(cat_roll.to_string(float_format=lambda x: f"{x:.1f}"))

# ------------ TP / FP / FN / TN contributions (by CV decision) ------------
def table_from_mask(mask, label):
    pct = contrib_pct(residual_sq, mask)
    out = pd.DataFrame({"feature": feat_names, f"{label}_%": pct})
    return out.sort_values(f"{label}_%", ascending=False).reset_index(drop=True)

tp_tbl = table_from_mask(m_tp, "TP")
fp_tbl = table_from_mask(m_fp, "FP")
fn_tbl = table_from_mask(m_fn, "FN")
tn_tbl = table_from_mask(m_tn, "TN")

# Quick view
all_series = pd.Series(contrib_all, index=feat_names, name="All_%")
def top_with_delta(df, label, k=10):
    out = df.head(k).merge(all_series.reset_index().rename(columns={"index":"feature"}),
                           on="feature", how="left")
    out[f"Delta_{label}_minus_All_pp"] = out[f"{label}_%"] - out["All_%"]
    print(f"\n=== Top {k} features — {label} ===")
    print(out.to_string(index=False, float_format=lambda x: f"{x:.2f}"))

# 🔄 Save full per-class feature tables (no top-k truncation)
all_series = pd.Series(contrib_all, index=feat_names, name="All_%")

for label, df in {"TP": tp_tbl, "FP": fp_tbl, "FN": fn_tbl, "TN": tn_tbl}.items():
    merged = df.merge(all_series.reset_index().rename(columns={"index": "feature"}),
                      on="feature", how="left")
    merged[f"Delta_{label}_minus_All_pp"] = merged[f"{label}_%"] - merged["All_%"]
    merged.to_csv(outdir / f"cae_cv_all_features_{label}.csv", index=False)
    print(f"✅ Saved: {outdir / f'cae_cv_all_features_{label}.csv'}")

top_with_delta(tp_tbl, "TP")
top_with_delta(fp_tbl, "FP")
top_with_delta(fn_tbl, "FN")
top_with_delta(tn_tbl, "TN")

# ----------------------------
# NEW: Movement–residual linkage (inverse scaling)
# ----------------------------
SCALER_PATH = fold_dirs[0] / "CV_scaler_infection_30_Conditional_exp.pkl"
try:
    scaler = joblib.load(SCALER_PATH)
    if not hasattr(scaler, "inverse_transform"):
        raise TypeError("Loaded scaler has no inverse_transform")

    num_cols = [c for c in feat_names if c in scaler.feature_names_in_]
    cat_cols = [c for c in feat_names if c not in num_cols]

    X_num = df_test[num_cols].to_numpy()
    X_cat = df_test[cat_cols].to_numpy() if cat_cols else np.empty((len(df_test), 0))
    X_num_raw = scaler.inverse_transform(X_num)

    df_raw = pd.DataFrame(np.hstack([X_num_raw, X_cat]), columns=num_cols + cat_cols)
except Exception as e:
    print(f"[warn] Using scaled units — {e}")
    df_raw = df_test.copy()

# Add IDs and masks
df_raw["id"] = rows_cae["id"].values
df_raw["row_flag_cae"] = rows_cae["row_flag_cae"].values

# Movement summaries
df_raw["speed"]   = df_raw.filter(like="velo").mean(axis=1)
df_raw["turning"] = df_raw.filter(like="angle").mean(axis=1)
df_raw["sinu"]    = df_raw.filter(like="sinu").mean(axis=1)

def group_summary(mask, label):
    tmp = df_raw.loc[mask]
    return pd.Series({
        "n_rows": len(tmp),
        "mean_speed": tmp["speed"].mean(),
        "mean_turning": tmp["turning"].mean(),
        "mean_sinu": tmp["sinu"].mean()
    }, name=label)

summaries = pd.concat([
    group_summary(m_tp, "TP"),
    group_summary(m_fp, "FP"),
    group_summary(m_fn, "FN"),
    group_summary(m_tn, "TN"),
    group_summary(mask_flag, "Flagged")
], axis=1).T

# % diffs vs TP
for col in ["mean_speed", "mean_turning", "mean_sinu"]:
    baseline = summaries.loc["TP", col]
    for grp in ["FP", "FN", "TN", "Flagged"]:
        summaries.loc[grp, f"{col}_vsTP_%diff"] = (
            (summaries.loc[grp, col] - baseline) / baseline * 100 if baseline != 0 else np.nan
        )

print("\n=== Movement summaries by class (raw units), %diff vs TP ===")
print(summaries.round(3))
summaries.to_csv(outdir / "cae_cv_movement_summaries_by_class.csv")

# Direct per-variable summaries
vars_of_interest = ["abs_rel_angle","sinuosity_28pt","velo",
                    "rolling_abs_rel_angle_mean","rolling_velo_sd"]
def group_summary_vars(mask, label):
    tmp = df_raw.loc[mask, vars_of_interest]
    stats = {f"{v}_mean": tmp[v].mean() for v in vars_of_interest}
    stats.update({f"{v}_sd": tmp[v].std() for v in vars_of_interest})
    stats["n_rows"] = len(tmp)
    return pd.Series(stats, name=label)

summaries_vars = pd.concat([
    group_summary_vars(m_tp,"TP"),
    group_summary_vars(m_fp,"FP"),
    group_summary_vars(m_fn,"FN"),
    group_summary_vars(m_tn,"TN"),
    group_summary_vars(mask_flag,"Flagged"),
], axis=1).T

# diffs vs TP
for v in vars_of_interest:
    baseline = summaries_vars.loc["TP", f"{v}_mean"]
    for grp in ["FP","FN","TN","Flagged"]:
        summaries_vars.loc[grp, f"{v}_diff_vsTP"] = summaries_vars.loc[grp, f"{v}_mean"] - baseline
        summaries_vars.loc[grp, f"{v}_pctdiff_vsTP"] = (
            (summaries_vars.loc[grp, f"{v}_mean"] - baseline) / baseline * 100 if baseline != 0 else np.nan
        )

print("\n=== Per-variable movement summaries by class ===")
print(summaries_vars.round(3))

# Residual correlation with movement metrics
resid_total = residual_sq.sum(axis=1)
movement_links = pd.DataFrame({
    "resid_total": resid_total,
    "speed": df_raw["speed"],
    "turning": df_raw["turning"],
    "sinu": df_raw["sinu"]
}).corr()

print("\n=== Correlation of movement variables with total reconstruction error ===")
print(movement_links.round(3))

# turn the string into a Path and make sure it exists
outdir = Path("<LOCAL_PATH>")
outdir.mkdir(parents=True, exist_ok=True)

# now this works:
summaries.to_csv(outdir / "cae_cv_movement_summaries_by_class.csv")
summaries_vars.to_csv(outdir / "cae_cv_movement_summaries_by_variable.csv")
movement_links.to_csv(outdir / "cae_cv_movement_residual_correlations.csv")


In [ ]:
from pathlib import Path

# -------------------- 1. Helper: same metrics function --------------------
def wilson_ci(k, n, z=1.96):
    if n == 0: return (np.nan, np.nan)
    p = k / n
    denom = 1 + z*z/n
    center = (p + z*z/(2*n)) / denom
    half = z * np.sqrt((p*(1-p)/n) + (z*z/(4*n*n))) / denom
    return (center - half, center + half)

def counts_to_metrics(TN, FP, FN, TP):
    N = TP + TN + FP + FN
    acc  = (TP + TN) / N if N else np.nan
    prec = TP / (TP + FP) if (TP + FP) else np.nan
    sens = TP / (TP + FN) if (TP + FN) else np.nan
    spec = TN / (TN + FP) if (TN + FP) else np.nan
    tss  = sens + spec - 1 if (np.isfinite(sens) and np.isfinite(spec)) else np.nan
    acc_lo, acc_hi   = wilson_ci(TP + TN, N) if N else (np.nan, np.nan)
    prec_lo, prec_hi = wilson_ci(TP, TP + FP) if (TP + FP) else (np.nan, np.nan)
    sens_lo, sens_hi = wilson_ci(TP, TP + FN) if (TP + FN) else (np.nan, np.nan)
    spec_lo, spec_hi = wilson_ci(TN, TN + FP) if (TN + FP) else (np.nan, np.nan)
    tss_lo, tss_hi   = (sens_lo + spec_lo - 1, sens_hi + spec_hi - 1) if (np.isfinite(sens_lo) and np.isfinite(spec_lo)) else (np.nan, np.nan)
    return dict(n_eval=N, TP=TP, FP=FP, FN=FN, TN=TN,
                accuracy=acc, accuracy_lo=acc_lo, accuracy_hi=acc_hi,
                precision=prec, precision_lo=prec_lo, precision_hi=prec_hi,
                tss=tss, tss_lo=tss_lo, tss_hi=tss_hi)

def t_crit_95(n):
    if n <= 1: return np.nan
    if n >= 30: return 2.042
    table = {2: 12.706, 3: 4.303, 4: 3.182, 5: 2.776, 6: 2.571, 7: 2.447, 8: 2.365, 9: 2.306, 10: 2.262, 12: 2.201, 15: 2.131, 20: 2.086}
    return table.get(n, 2.262)

# -------------------- 2. ENSEMBLE METRICS --------------------
sub_ens = ids_df[ids_df["eligible"] == 1]
y_true_ens = sub_ens["true_label_id"].astype(int).to_numpy()
y_pred_ens = sub_ens["id_pred"].astype(int).to_numpy()
TN, FP, FN, TP = confusion_matrix(y_true_ens, y_pred_ens, labels=[0,1]).ravel()
metrics_ens = counts_to_metrics(TN, FP, FN, TP)
metrics_ens["mode"] = "ensemble"

# -------------------- 3. ACROSS-FOLD METRICS --------------------
fold_idxs = sorted(int(re.search(r"score_f(\d+)", c).group(1)) for c in ids_df.columns if re.match(r"score_f\d+", c))
acc_list, prec_list, tss_list = [], [], []

for f in fold_idxs:
    score_col = f"score_f{f}"
    elig_col = f"eligible_f{f}"
    tau_col = f"tau_f{f}"

    if elig_col not in ids_df.columns or score_col not in ids_df.columns:
        continue

    fold_sub = ids_df[ids_df[elig_col] == 1].copy()
    if tau_col in fold_sub.columns:
        tau_vals = fold_sub[tau_col]
    else:
        tau_vals = ids_df[tau_col].dropna().iloc[0] if tau_col in ids_df.columns else 0.5
    fold_sub["id_pred_f"] = (fold_sub[score_col] > tau_vals).astype(int)

    y_true_f = fold_sub["true_label_id"].astype(int).to_numpy()
    y_pred_f = fold_sub["id_pred_f"].astype(int).to_numpy()
    TN, FP, FN, TP = confusion_matrix(y_true_f, y_pred_f, labels=[0,1]).ravel()
    met = counts_to_metrics(TN, FP, FN, TP)

    acc_list.append(met["accuracy"])
    prec_list.append(met["precision"])
    tss_list.append(met["tss"])

# Aggregate
def summarize(vals):
    vals = np.asarray(vals)
    vals = vals[np.isfinite(vals)]
    if len(vals) == 0:
        return (np.nan, np.nan, np.nan)
    mean = float(np.mean(vals))
    std = float(np.std(vals, ddof=1)) if len(vals) > 1 else 0.0
    ci = t_crit_95(len(vals)) * std / np.sqrt(len(vals)) if len(vals) > 1 else np.nan
    return (mean, mean - ci if np.isfinite(ci) else np.nan, mean + ci if np.isfinite(ci) else np.nan)

acc_m, acc_lo, acc_hi = summarize(acc_list)
prec_m, prec_lo, prec_hi = summarize(prec_list)
tss_m, tss_lo, tss_hi = summarize(tss_list)

metrics_fold = {
    "mode": "across_folds",
    "n_eval": len(ids_df[ids_df["eligible"] == 1]),
    "TP": TP, "FP": FP, "FN": FN, "TN": TN,
    "accuracy": acc_m, "accuracy_lo": acc_lo, "accuracy_hi": acc_hi,
    "precision": prec_m, "precision_lo": prec_lo, "precision_hi": prec_hi,
    "tss": tss_m, "tss_lo": tss_lo, "tss_hi": tss_hi
}

# -------------------- 4. Save to CSV --------------------
results_df = pd.DataFrame([metrics_ens, metrics_fold])
results_csv = base / "CV_metrics_summary.csv"
results_df.to_csv(results_csv, index=False)
print(f"✅ Saved summary metrics → {results_csv}")


## New Post Analysis

In [ ]:
# 1) Reload immediately to verify
df2 = pd.read_csv(output_csv_path)

# 2) Recompute high_ids from the file
file_high_ids = df2.loc[df2['predicted_CWD_status'] == 'positive', 'id'].unique()
print("high_ids (in memory):", high_ids)
print("high_ids (in file)  :", file_high_ids)

# 3) For each of those IDs, show how many rows were flagged positive
for i in file_high_ids:
    c = df2[(df2['id']==i) & (df2['predicted_CWD_status']=='positive')].shape[0]
    print(f"ID {i} → {c} rows predicted positive")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 1) Load the enriched results CSV (must include id, predicted_CWD_status, err_neg, err_pos, err_pred)
df = pd.read_csv("<LOCAL_PATH>")

# 2) Annotate the true CWD status for those 22 IDs
actual_pos = {16, 18, 19, 21, 31, 53, 58, 72, 92, 102}
actual_neg = {3, 4, 7, 34, 38, 42, 43, 55, 56, 60, 77, 78, 82, 89, 95, 100, 111, 115, 119, 120, 129}

df['true_num'] = df['id'].map(lambda i: 1 if i in actual_pos else 0)
df['pred_num'] = df['predicted_CWD_status'].map({'negative':0,'positive':1})

# 3) Aggregate to one call per ID (majority‐vote)
id_summary = df.groupby('id').agg(
    true_label   = ('true_num', 'first'),
    positive_frac= ('pred_num', 'mean'),
    mean_err_pred= ('err_pred', 'mean'),
    mean_err_neg = ('err_neg',  'mean'),
    mean_err_pos = ('err_pos',  'mean'),
    n_points     = ('pred_num', 'count')
).reset_index()

# label 1 if >=50% of points were called positive
id_summary['pred_label'] = (id_summary['positive_frac'] >= 0.01).astype(int)

# 4) Compute TP/FP/FN/TN
TP = int(((id_summary['true_label']==1) & (id_summary['pred_label']==1)).sum())
TN = int(((id_summary['true_label']==0) & (id_summary['pred_label']==0)).sum())
FP = int(((id_summary['true_label']==0) & (id_summary['pred_label']==1)).sum())
FN = int(((id_summary['true_label']==1) & (id_summary['pred_label']==0)).sum())

print(f"TP={TP}, FP={FP}, FN={FN}, TN={TN}")
print(f"Accuracy={(TP+TN)/(TP+TN+FP+FN):.2f}")
print(f"FPR={FP/(FP+TN):.2f}, FNR={FN/(FN+TP):.2f}")

# 5) Show which IDs were false positives
fp_ids = id_summary.loc[(id_summary.true_label==0)&(id_summary.pred_label==1), 'id'].tolist()
print("False positive IDs:", fp_ids)

# 6) Inspect their error profiles
print("\nFalse Positive breakdown:")
print(id_summary[id_summary['id'].isin(fp_ids)][[
    'id','n_points','mean_err_pred','mean_err_neg','mean_err_pos'
]])

# 7) Plot error distributions by true vs. false positives
plt.figure()
plt.hist(id_summary[id_summary.true_label==1]['mean_err_pred'], bins=10, edgecolor='black', alpha=0.7)
plt.title("Avg err_pred for true positives")
plt.xlabel("mean_err_pred"); plt.ylabel("Count"); plt.tight_layout(); plt.show()

plt.figure()
plt.hist(id_summary[id_summary.true_label==0]['mean_err_pred'], bins=10, edgecolor='black', alpha=0.7)
plt.title("Avg err_pred for true negatives")
plt.xlabel("mean_err_pred"); plt.ylabel("Count"); plt.tight_layout(); plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# ── 1) Paths ───────────────────────────────────────────────────────────
test_feat_path = "<LOCAL_PATH>"
pred_path      = "<LOCAL_PATH>"

# ── 2) Load data ────────────────────────────────────────────────────────
# Make sure date_time is parsed as datetime
df_feat = pd.read_csv(test_feat_path, parse_dates=['date_time'])
df_pred = pd.read_csv(pred_path)

# ── 3) Merge on both id AND date_time ───────────────────────────────────
df = pd.merge(
    df_feat,
    df_pred[['id','err_pred','err_neg','err_pos','predicted_CWD_status']],
    on='id',
    how='inner'
)

# If your merge produces duplicate rows, you can also merge on index:
# df = df_feat.join(df_pred.set_index(df_feat.index), rsuffix='_pred')

# ── 4) Filter to the two false-positive IDs ────────────────────────────
fp_ids = [16, 18, 19, 21, 31, 53, 58, 60, 72, 92, 102]
df_fp = df[df['id'].isin(fp_ids)].sort_values(['id','date_time'])

# ── 5) Plot reconstruction error over time ─────────────────────────────
for deer_id in fp_ids:
    sub = df_fp[df_fp['id'] == deer_id]
    plt.figure()
    plt.plot(sub['date_time'], sub['err_pred'])
    plt.title(f"ID {deer_id}: Reconstruction Error Over Time")
    plt.xlabel("Date")
    plt.ylabel("err_pred")
    plt.tight_layout()
    plt.show()

# ── 6) (Optional) Plot raw feature trends — replace FEATURE_1/2 below ────
FEATURES = [    'velo',
    'abs_rel_angle',
    'sinuosity_28pt',
    'rolling_abs_rel_angle_mean',
    'rolling_abs_rel_angle_sd',
    'rolling_abs_rel_angle_skew',
    'rolling_abs_rel_angle_kurt',
    'rolling_velo_mean',
    'rolling_velo_sd',
    'rolling_velo_skew',
    'rolling_velo_kurt',
    'low',
    'est',
    'high'] 
for deer_id in fp_ids:
    sub = df_fp[df_fp['id'] == deer_id]
    plt.figure()
    for feat in FEATURES:
        plt.plot(sub['date_time'], sub[feat], label=feat)
    plt.title(f"ID {deer_id}: Feature Trends Over Time")
    plt.xlabel("Date")
    plt.legend()
    plt.tight_layout()
    plt.show()


## Common Trend is the abs_rel_angle

## Investigating Anomalies

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# === Load the anomaly results CSV ===
csv_path = "<LOCAL_PATH>"
df = pd.read_csv(csv_path)

# === Define your CWD ID categories ===
actual_cwd_instances = [1, 2, 6, 8, 11, 12, 15, 16, 17, 18, 19, 20, 21, 23, 24, 25, 30, 31, 32, 35, 36, 37, 40, 47, 48, 53, 57, 58, 61, 62, 64, 65, 67, 68, 69, 70, 71, 72, 73, 74, 79, 80, 83, 90, 91, 92, 93, 94, 96, 102, 105, 107, 113, 114, 116, 121, 122, 125, 130, 141]
unaffected_cwd_instances = []
high_anomaly_ids = [1, 2, 6, 8, 11, 12, 15, 16, 17, 18, 19, 20, 21, 23, 24, 25, 30, 31, 32, 35, 36, 37, 40, 47, 48, 53, 57, 58, 61, 62, 64, 65, 67, 68, 69, 70, 71, 72, 74, 79, 80, 83, 90, 91, 92, 93, 94, 96, 102, 105, 107, 113, 114, 116, 121, 122, 130, 141]

# === Assign case type per ID ===
def classify_id(row):
    id_ = row['id']
    if id_ in actual_cwd_instances:
        return "TP" if id_ in high_anomaly_ids else "FN"
    elif id_ in unaffected_cwd_instances:
        return "FP" if id_ in high_anomaly_ids else "TN"
    else:
        return "Unknown"

df['case_type'] = df.apply(classify_id, axis=1)

# === Summary statistics by case_type ===
# Keep only numeric columns (excluding ID just in case it's numeric)
numeric_cols = df.select_dtypes(include='number').columns.tolist()
summary_stats = df[numeric_cols + ['case_type']].groupby('case_type').agg(['mean', 'std'])

# Display stats for reconstruction error
print(summary_stats['reconstruction_error'])


# === Boxplot of a feature by case_type ===
plt.figure(figsize=(10, 6))
sns.boxplot(data=df[df['case_type'] != 'Unknown'], x='case_type', y='reconstruction_error')
plt.title("Reconstruction Error by Case Type")
plt.tight_layout()
plt.show()




In [ ]:
# Create subsets
df_fp = df[df['case_type'] == 'FP']
df_fn = df[df['case_type'] == 'FN']
df_tp = df[df['case_type'] == 'TP']
df_tn = df[df['case_type'] == 'TN']

# Get numeric feature columns
numeric_cols = df.select_dtypes(include='number').columns.difference(['id', 'reconstruction_error', 'is_anomaly'])

# Mean difference between FN and TP
fn_tp_diff = df_fn[numeric_cols].mean() - df_tp[numeric_cols].mean()
print("\nTop features where FN differs from TP:")
print(fn_tp_diff.abs().sort_values(ascending=False).head(10))


# Choose one top feature to visualize
top_fn_feat = fn_tp_diff.abs().sort_values(ascending=False).index[0]

# FN vs TP
plt.figure(figsize=(10, 4))
sns.boxplot(data=df[df['case_type'].isin(['FN', 'TP'])], x='case_type', y=top_fn_feat)
plt.title(f"{top_fn_feat}: FN vs TP")
plt.show()



print("\nCorrelation with error in FNs:")
print(df_fn.corr(numeric_only=True)['reconstruction_error'].sort_values(ascending=False).head(10))


## Correlation does not mean causation, but provides some interesting results regarding masking movement during different seasons

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# --- Compute per-feature residuals ---
residuals = X_test - reconstructed  # shape: (n_samples, n_features)
abs_residuals = np.abs(residuals)

# --- Load the test set with metadata and case_type ---
df_meta = pd.read_csv("<LOCAL_PATH>")
df_meta['case_type'] = df_meta.apply(classify_id, axis=1)

# --- Match residuals back to DataFrame ---
feature_names = df_meta.select_dtypes(include='number').columns.difference(['id', 'reconstruction_error', 'is_anomaly'])
df_resid = pd.DataFrame(abs_residuals, columns=feature_names)
df_combined = pd.concat([df_meta[['id', 'case_type']].reset_index(drop=True), df_resid], axis=1)
# Drop ID before calculating mean residuals
df_combined_clean = df_combined.drop(columns=['id'])

# Group and compute means without ID
group_means = df_combined_clean.groupby('case_type').mean()


# --- Compare FN vs TP residuals ---
fn_vs_tp_diff = group_means.loc['FN'] - group_means.loc['TP']
top_fn_features = fn_vs_tp_diff.abs().sort_values(ascending=False).head(10)
print("\nTop features with highest residuals in FN vs TP:")
print(top_fn_features)


# --- Plot example: top FN feature ---
top_fn_feat = top_fn_features.index[0]
plt.figure(figsize=(8, 4))
sns.boxplot(data=df_combined[df_combined['case_type'].isin(['FN', 'TP'])], x='case_type', y=top_fn_feat)
plt.title(f"Absolute Residual for {top_fn_feat} (FN vs TP)")
plt.tight_layout()
plt.show()


# Key Interpretations (based on updated results + visuals)

## False Positives (FP) vs True Negatives (TN)
### Top residual features: season_winter, abs_rel_angle, rolling_r2n_var, and rolling_sinuosity_mean.
### The residual for season_winter is quite small (~0.0038), but visible in the boxplot — there’s a consistent, minor misfit in how winter data gets reconstructed.
## Implication:
### Falsely flagged healthy animals likely exhibit mild but consistent winter-specific deviations in direction and turning behavior. While not dramatic enough to be true anomalies, ### they diverge just enough from what the autoencoder learned during training (possibly dominated by non-winter patterns).
## False Negatives (FN) vs True Positives (TP)
### The largest residual differences are now much more pronounced, especially:
### rolling_sinuosity_mean (0.04)
### rolling_sinuosity (0.021)
## Boxplot confirms that FNs have systematically lower residuals than TPs for these features.
## Implication:
### FNs did show some abnormal movement (especially sinuosity), but:
### The error was too uniformly small to cross the anomaly threshold.
### They may represent early-stage or low-expression disease cases, or animals with slow/progressive behavioral changes. (Get Dr. Cunningham's opinion on this interpretation)


## Had realisation here that we have be having false negative due to short temporal range and updated the testing.